# models

> The models: the model for each task, the PII guard for the text that leaves the Mac, and speech.

This module holds the models that abhyasa talks to. It imports only from `abhyasa.learner`. It has three parts:

- **The model for each task**: the route of each task (`for_task`), the default ask (`route_ask`), the list of models, and where each one runs.
- **The guard**: ramabana's PII guard. Text that leaves the Mac passes it first.
- **Speech**: speech to text and text to speech, on this Mac or with OpenAI.

The heavy model code (ramabana, rishi, mlx-audio) loads inside the functions that use it, so the import of this module stays light.

In [ ]:
#| default_exp models

In [ ]:
#| hide
import os, tempfile
os.environ.setdefault('LITESEARCH_CACHE', tempfile.mkdtemp(prefix='ls-'))   # and litesearch does not wait on the shared lock of another run
os.environ['ABHYASA_CONFIG'] = tempfile.mkdtemp()   # the settings go to a temporary folder, not to your real settings
os.environ['RAMABANA_CONFIG'] = os.path.join(tempfile.mkdtemp(), 'config.toml')   # and ramabana reads no real config
os.environ['VISHALAKSHI_VAULT'] = os.environ['ABHYASA_VAULT'] = os.path.join(tempfile.mkdtemp(), 'vault.db')   # a test that opens the default vault gets a temporary one

In [ ]:
#| hide
from fastcore.test import *

In [ ]:
#| hide
import ramabana.core
ramabana.core.PROBE_DIR = tempfile.mkdtemp(prefix='probes-')   # and the model list goes to a temporary folder, not to the real probe files
for k in ('ABHYASA_STT', 'OPENAI_API_KEY'): os.environ.pop(k, None)

Import the modules that this module uses. The heavy ones load inside the functions that need them, so the import stays light.

In [ ]:
#| export
import json, os, re, gc, time, uuid, inspect, tempfile, threading, wave, contextvars
from fastcore.utils import *
from abhyasa.learner import val, dflt, fit, settings, SETTINGS, BORROWS, CtxPool
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

## The model for each task

One call shape for each model: `ask(prompt, sp) -> str`.

This module connects the tutor to a model. The tutor needs only one function, `ask(prompt, sp)`, which returns text.
The module also parses JSON replies and gives `Fake`, a model for tests that runs offline.

Import the tools that this module uses.

### The model

Each surface gives its own `ask`.

| surface | `ask` |
|---|---|
| web app and notebook | `route_ask`: each call goes through the routing of ramabana, on a backend that stays open |
| the ramabana terminal agent | `agent.oneshot`, so the routing of the agent applies |
| tests | `Fake` |

The default model is `--model`, else `$ABHYASA_MODEL`, else `DFLT_MODEL` (`claude/sonnet`).

In [ ]:
#| export
DFLT_MODEL = 'claude/sonnet'

def dflt_model(model:str=None) -> str:
    "`model`, else `$ABHYASA_MODEL`, else the setting `models.default`, else `DFLT_MODEL`."
    return model or os.getenv('ABHYASA_MODEL') or val('models.default') or DFLT_MODEL

### JSON replies

`parse_json` finds the JSON in a reply. `ask_json` asks again one time when the reply is not valid JSON.

In [ ]:
#| export
def parse_json(s:str):
    "The first JSON object or array in `s`, also inside a ``` fence. Raise `ValueError` when there is none."
    s = s.strip()
    dec = json.JSONDecoder()
    # the whole reply first, then each fence, then the first object or array in the text: a fence inside a JSON string stays whole
    for t in [s, *(m.strip() for m in re.findall(r'```(?:json)?\s*(.*?)```', s, re.S))]:
        try: return json.loads(t)
        except json.JSONDecodeError: pass
    for i, ch in enumerate(s):
        if ch in '{[':
            try: return dec.raw_decode(s, i)[0]
            except json.JSONDecodeError: continue
    raise ValueError(f'no JSON in the reply: {s[:200]!r}')

def _check(r, need):
    "`r`, when it is a JSON object with text in each key of `need`; else a ValueError that says what is missing."
    if need and not (isinstance(r, dict) and all(isinstance(r.get(k), str) and r[k].strip() for k in need)):
        raise ValueError(f"the reply must be one JSON object with text in {', '.join(need)}")
    return r

def ask_json(ask, prompt:str, sp:str='', tries:int=2, need=()):
    "Ask for JSON and parse it. On a bad reply, or one with no text in a key of `need`, ask one more time with the error."
    p = prompt + '\n\nReply with only one JSON value and no other text.'
    for i in range(tries):
        out = ask(p, sp)
        try: return _check(parse_json(out), need)
        except ValueError as e:
            if i == tries - 1: raise
            p = f'{prompt}\n\nYour last reply was not valid JSON ({e}). Reply with only one JSON value.'

`parse_json` reads JSON in a fence, JSON inside other text, and arrays. A fence inside a JSON string does not break it. It fails when there is no JSON.

In [ ]:
test_eq(parse_json('```json\n{"a": 1}\n```'), {'a': 1})
test_eq(parse_json('Here it is: {"a": [1, 2]} thanks'), {'a': [1, 2]})
test_eq(parse_json('[{"x": 1}]'), [{'x': 1}])
test_fail(lambda: parse_json('nothing'), contains='no JSON')
test_eq(parse_json('{"answer": "Use ```python\\nx = 1\\n``` here."}'), {'answer': 'Use ```python\nx = 1\n``` here.'})
test_eq(parse_json('Sure. {"a": 1} and then {"b": 2}'), {'a': 1})

### A model for tests

`Fake` replies from a dict of rules. The first rule with a key in the prompt gives the reply.
It sends a dict or a list reply as JSON. It keeps each call in `calls`.

In [ ]:
#| export
class Fake:
    "A stand-in model for tests: `Fake({'grade': {...}, 'question': {...}})`."
    def __init__(self, rules:dict=None, dflt='{}'): self.rules, self.dflt, self.calls = rules or {}, dflt, []
    def __call__(self, prompt, sp=''):
        self.calls.append((prompt, sp))
        r = next((v for k, v in self.rules.items() if k in prompt or k in sp), self.dflt)
        if callable(r): r = r(prompt, sp)
        return r if isinstance(r, str) else json.dumps(r)

A matching rule gives its reply, and `Fake` records the call.

In [ ]:
f = Fake({'GRADE': dict(correct=True)})
test_eq(ask_json(f, 'GRADE this'), dict(correct=True))
test_eq(len(f.calls), 1)

After a reply that is not JSON, `ask_json` asks again and gets the JSON.

In [ ]:
bad = Fake(dict(X=lambda p, sp: 'oops' if 'not valid' not in p else '{"ok": 1}'))
test_eq(ask_json(bad, 'X'), {'ok': 1})

With `need`, a reply must be one JSON object with text in each of those keys. A list, or an object with no question, gets one more try. A second bad reply raises a `ValueError` that names the key.

In [ ]:
odd = Fake(dict(Q=lambda p, sp: '{"question": "What is a rank?"}' if 'must be one JSON object' in p else '[1, 2]'))
test_eq(ask_json(odd, 'Q', need=('question',)), {'question': 'What is a rank?'})
test_fail(lambda: ask_json(Fake(dflt={}), 'Q', need=('question',)), contains='question')

### A model for each task

The tutor does five tasks, and each task can use its own model. A quick model can grade and give hints. A strong model can
write questions and lessons. The Settings page saves the choice. `Router` reads it on each call, so a change has an effect on the next call.

| task | what the model does |
|---|---|
| `question` | writes a question and its hidden model answer |
| `grade` | reads the conversation and grades it |
| `hint` | gives the next hint |
| `lesson` | teaches the smallest step after a miss |
| `extract` | finds the concepts in a text |

Each task also has a reply cap: the most tokens that the model can write in one reply. `TOKENS` gives the default of each task.
A question has options and a rubric, and an extract can name 15 concepts, so these two get the most. A hint is short. The Settings
page can change each cap. Only the cap is a setting, because ramabana's `oneshot` uses no other per-call value: it turns thinking
off, and the time limit of a model is in ramabana's `config.toml`. The model layer gives the cap to the API models (OpenAI, Codex,
Gemini, Copilot) and to the models on this Mac. Claude Code does not use it for the reply, but ramabana uses it to fit the prompt
in the window of the model.

In [ ]:
#| export
TASKS = ('question', 'grade', 'hint', 'lesson', 'extract')
TOKENS = {t: dflt(f'tokens.{t}') for t in TASKS}   # the reply cap of each task, when the settings give none
TOKENS_MAX = SETTINGS['tokens.question'].hi

`model_ask` gives an `ask` for any model name. The name can be any model that ramabana can run: Claude Code, Codex, OpenAI, Copilot,
Gemini, or a model on this machine (LiteRT, mlx, llama.cpp, Ollama). The call goes through the routing of ramabana, as `route_ask` below shows.

In [ ]:
#| export
import threading, time
_asks, _asks_lock = {}, threading.Lock()

def model_ask(name:str):
    "An `ask(prompt, sp='')` for model `name`, made once and kept."
    with _asks_lock:
        if name not in _asks: _asks[name] = route_ask(name)
        return _asks[name]

### Names for the model list

The Settings page shows a list of models. Each item has a name that people know, such as "Claude Sonnet" or "GPT-5.5".
The model id, such as `claude/sonnet`, is the value that the settings keep. ramabana's `model_label` makes the name from the id,
and `model_label` here adds "a local model:" before the name of a model that runs on this Mac.
`PROVIDERS` gives the name of each group in the list, and `provider_label` reads it. ramabana's `is_local` tells if a model runs on this Mac.
For Ollama, it looks at the daemon host (the `host` in config.toml, else `$OLLAMA_HOST`), and it does not connect to it.
Thus a model on an Ollama daemon on another computer has no "a local model" name, and its group is "Ollama on another computer".

In [ ]:
#| export
PROVIDERS = dict(claude='Claude', anthropic='Anthropic API', openai='OpenAI', codex='Codex', gemini='Gemini', copilot='GitHub Copilot',
                 azure='Azure', remote='Other hosted models', litert='On this Mac (LiteRT)', mlx='On this Mac (MLX)',
                 llama='On this Mac (llama.cpp)', ollama='On this Mac (Ollama)')   # the group names of the Settings list
LOCAL = 'a local model: '

def _on_mac(value:str, provider:str='') -> bool:
    "Whether the model `value` of `provider` runs on this Mac: ramabana's `is_local`."
    from ramabana.core import is_local
    return is_local(value) or bool(provider) and is_local(f'{provider}/{value}')

def model_label(value:str, provider:str='', repo:str='') -> str:
    "ramabana's `model_label` of the model id `value`, with `LOCAL` before a model on this Mac; `repo` is the download of a local model, if known."
    from ramabana.models import model_label as label
    if provider in ('remote', 'azure'): return value   # a name that you gave in ramabana's config.toml
    name = label(value if '/' in value or not provider else f'{provider}/{value}', repo)
    return LOCAL + name if _on_mac(value, provider) else name

def provider_label(provider:str, values=()) -> str:
    "The name of the `provider` group in the model list, for the model ids `values`. An Ollama daemon on another computer is not on this Mac."
    if provider == 'ollama' and values:
        here = {_on_mac(v, provider) for v in values}
        if here == {False}: return 'Ollama on another computer'
        if here == {True, False}: return 'Ollama'
    return PROVIDERS.get(provider, provider.capitalize() or 'Other')

Each kind of id gets a plain name. A Claude Code tier gets the word Claude. A local model says so.

In [ ]:
test_eq(model_label('claude/sonnet'), 'Claude Sonnet')
test_eq(model_label('claude/claude-opus-5-5'), 'Claude Opus 5.5')
test_eq(model_label('anthropic/claude-haiku-4-5-20251001'), 'Claude Haiku 4.5')
test_eq(model_label('codex/gpt-6-astra'), 'GPT-6 Astra')
test_eq(model_label('openai/gpt-5.4-mini'), 'GPT-5.4 Mini')
test_eq(model_label('gemini/gemini-3.5-flash'), 'Gemini 3.5 Flash')
test_eq(model_label('qwen-4b', 'mlx', 'mlx-community/Qwen3.5-4B-MLX-4bit'), 'a local model: Qwen 3.5 4B')
test_eq(model_label('gemma-e4b', 'litert', 'litert-community/gemma-4-E4B-it-litert-lm'), 'a local model: Gemma 4 E4B')
test_eq(model_label('my-model', 'azure'), 'my-model')

An OpenAI id gets the name that OpenAI uses. A snapshot date such as `2024-07-18` goes, so the name is the same as the name of the model.

In [ ]:
test_eq(model_label('openai/gpt-4o-mini-2024-07-18'), 'GPT-4o mini')
test_eq(model_label('copilot/gpt-4-o-preview'), 'GPT-4o preview')
test_eq(model_label('openai/gpt-4o'), 'GPT-4o')
test_eq(model_label('openai/gpt-4.1-mini'), 'GPT-4.1 Mini')

The o-series names stay in lower case, with their hyphen:

In [ ]:
test_eq(model_label('codex/o4-mini'), 'o4-mini')
test_eq(model_label('openai/o3'), 'o3')

One letter and a number stay together, as the maker writes them:

In [ ]:
test_eq(model_label('copilot/kimi-k3'), 'Kimi K3')
test_eq(model_label('openrouter/qwen3-8b'), 'Qwen 3 8B')   # a longer word and a number still get a space

An Ollama model is local only when its daemon is on this Mac. With `$OLLAMA_HOST` on localhost, the model and its group are on this Mac:

In [ ]:
from fastcore.xtras import modified_env

In [ ]:
with modified_env(OLLAMA_HOST='localhost:11434'):
    test_eq(model_label('ollama/qwen3:8b'), 'a local model: Qwen 3 8B')
    test_eq(provider_label('ollama', ['ollama/qwen3:8b']), 'On this Mac (Ollama)')

With `$OLLAMA_HOST` on another computer, the model is not called local, and its group has a hosted name. The check reads only the host name: it does not connect:

In [ ]:
with modified_env(OLLAMA_HOST='http://gpu-box.lan:11434'):
    test_eq(model_label('ollama/qwen3:8b'), 'Qwen 3 8B')
    test_eq(provider_label('ollama', ['ollama/qwen3:8b']), 'Ollama on another computer')

The other groups keep their names from `PROVIDERS`. A provider that is not in it gets its own name:

In [ ]:
test_eq(provider_label('mlx', ['qwen-4b']), 'On this Mac (MLX)')
test_eq(provider_label('claude', ['claude/sonnet']), 'Claude')
test_eq(provider_label('groq'), 'Groq')
test_eq(provider_label(''), 'Other')

#### Where a model runs

`where_of` says where the model `name` runs: `on this Mac` or `hosted`, and the company of a hosted model, from ramabana's `vendor_of`.
It uses `_on_mac`, as `provider_label` does, so an Ollama daemon on another computer is hosted. When ramabana does not know the company,
the name of the group stands in. An unknown model is hosted, as the guard takes it. `where_chip` gives the short text for the page.

In [ ]:
#| export
HERE = 'on this Mac'

def where_of(name:str) -> dict:
    "Who runs the model `name`: `model` (its label), `where` (`on this Mac` or `hosted`) and `who` (the company of a hosted model, ramabana's `vendor_of`). An unknown model is hosted, as the guard takes it."
    if not name: return dict(where='hosted')
    if _on_mac(name): return dict(model=model_label(name).removeprefix(LOCAL), where=HERE, who='')
    from ramabana.models import vendor_of
    pre = name.rpartition('/')[0]
    return dict(model=model_label(name), where='hosted', who=vendor_of(name) or (provider_label(pre, [name]) if pre else ''))

def where_chip(name:str) -> str:
    "`on this Mac`, or `hosted · <company>`, for the model `name`."
    w = where_of(name)
    return HERE if w['where'] == HERE else ' · '.join(x for x in ('hosted', w.get('who')) if x)

A Claude model is hosted by Anthropic. A model on an Ollama daemon on this Mac stays here.

In [ ]:
test_eq(where_chip('claude/haiku'), 'hosted · Anthropic')
test_eq(where_chip('copilot/gpt-6-sol'), 'hosted · GitHub')
with modified_env(OLLAMA_HOST='localhost:11434'): test_eq(where_chip('ollama/qwen3:8b'), HERE)
with modified_env(OLLAMA_HOST='http://gpu-box.lan:11434'): test_eq(where_chip('ollama/qwen3:8b'), 'hosted · Ollama on another computer')
test_eq(where_of(''), dict(where='hosted'))

`catalog` lists the models that can run here, for the Settings page: the rows of ramabana's `available_models(ttl=)`. Each row has a
`name` from ramabana's `model_label`, `local`, `vendor` and `model_id`. abhyasa adds "a local model:" before the name of a local row.
The look-up asks each provider, so it takes some seconds. ramabana keeps the list in memory and on the disk: a new process shows the
last list at once, and a list older than `ttl` seconds refreshes in the background. When the look-up fails, the list has only the models in `FALLBACK`.

In [ ]:
#| export
FALLBACK = ['claude/sonnet', 'claude/haiku', 'claude/opus']

def catalog(ttl:float=300.) -> list:
    "The models that can run here: ramabana's rows (`value`, `name`, `label`, `provider`, `source`, `local`, `vendor`, `model_id`), with `LOCAL` before the name of a local row. The last list comes at once; one older than `ttl` seconds refreshes in the background."
    try:
        from ramabana.core import available_models
        return [dict(r, name=LOCAL + r['name']) if r.get('local') else r for r in available_models(ttl=ttl)]
    except Exception: return [dict(value=m, label=m, provider='claude', source='rishi', name=model_label(m)) for m in FALLBACK]

These tests use ramabana. `slow` is a stand-in for the slow look-up of ramabana (`_model_rows`): it counts its calls, and gives one row with the facts of ramabana (`_facts`). The tests keep the list in a temp folder.

In [ ]:
import ramabana.core as rc

Two calls to `catalog` ask the providers one time.

In [ ]:
calls, real, real_dir = [], rc._model_rows, rc.PROBE_DIR
def slow(include_legacy=False): calls.append(1); time.sleep(.5); return [rc._facts(dict(value='claude/sonnet', label='sonnet', provider='claude', source='login'))]
rc._model_rows, rc.PROBE_DIR = slow, Path(tempfile.mkdtemp())
rc.forget_probes()
test_eq([r['name'] for r in catalog()], ['Claude Sonnet'])
test_eq((catalog(), len(calls))[1], 1)

A new process has no list in memory. It gets the last list from the disk at once, with no slow call, and a refresh starts in the background.

In [ ]:
rc.forget_probes()   # the memory copy goes, as in a new process; the disk copy stays
t = time.time(); rows = catalog()
assert time.time() - t < .3 and [r['name'] for r in rows] == ['Claude Sonnet']
for _ in range(50):
    if len(calls) == 2: break
    time.sleep(.05)
test_eq(len(calls), 2)

When the look-up fails, the list has the three Claude tiers, each with a name.

In [ ]:
def broken(include_legacy=False): raise OSError('no network')
rc._model_rows = broken; rc.forget_probes(disk=True)
test_eq([r['name'] for r in catalog()], ['Claude Sonnet', 'Claude Haiku', 'Claude Opus'])
rc._model_rows, rc.PROBE_DIR = real, real_dir; rc.forget_probes()

A local row gets "a local model:" before its name. A hosted row keeps the name of ramabana.

In [ ]:
row = lambda v, local: dict(value=v, label=v, provider='mlx' if local else 'claude', source='test', name='Qwen 4B' if local else 'Claude Haiku', local=local)
real_am = rc.available_models
rc.available_models = lambda include_legacy=False, ttl=None: [row('qwen-4b', True), row('claude/haiku', False)]
try: test_eq([r['name'] for r in catalog()], ['a local model: Qwen 4B', 'Claude Haiku'])
finally: rc.available_models = real_am

#### One row for each model

Some models come through two providers. For example, Claude Haiku 5.5 comes through Claude Code and through GitHub Copilot, and GPT-6.1 Sol
through Codex, GitHub Copilot and the OpenAI API. Each route has its own id, so the list showed the same name two or three times.
`one_each` keeps one row for each name: the route of the first provider in ramabana's `READY_ORDER` (subscriptions first, then keys).
The row keeps the other ids in `also`. A saved id (`keep`) always stays, so the list shows the choice that you made.

In [ ]:
#| export
def _rank(p:str) -> int:
    from ramabana.core import READY_ORDER
    return READY_ORDER.index(p) if p in READY_ORDER else len(READY_ORDER)

def one_each(rows:list, keep:str='') -> list:
    "One row of `rows` for each model name, in the first place of that name: the saved id `keep`, else the route that ramabana tries first. `also` has the other ids. An alias with a version beside it goes into the `also` of the newest version."
    key = lambda r: r.get('name') or r['value']
    best = {}
    for r in rows:
        b = best.get(key(r))
        if b is None or r['value'] == keep or b['value'] != keep and _rank(r.get('provider', '')) < _rank(b.get('provider', '')): best[key(r)] = r
    out, done = [], set()
    for r in rows:
        if (k := key(r)) in done: continue
        done.add(k)
        out.append(dict(best[k], also=[x['value'] for x in rows if key(x) == k and x is not best[k]]))
    for a in [r for r in out if r['value'] != keep]:
        vs = [r for r in out if re.fullmatch(re.escape(key(a)) + r' \d+(?:\.\d+)*', key(r))]
        if not vs: continue
        v = max(vs, key=lambda r: tuple(map(int, key(r).rpartition(' ')[2].split('.'))))   # the alias points to the newest version
        out.remove(a)
        v['also'] = v['also'] + [a['value'], *a['also']]
    return out

Claude Haiku 5.5 shows one time, on Claude Code. With the Copilot id saved, the Copilot row stays.

In [ ]:
dup = [dict(value='copilot/claude-haiku-5.5', name='Claude Haiku 5.5', provider='copilot'), dict(value='claude/sonnet', name='Claude Sonnet', provider='claude'),
       dict(value='claude/claude-haiku-5-5', name='Claude Haiku 5.5', provider='claude')]
test_eq([(r['value'], r['also']) for r in one_each(dup)], [('claude/claude-haiku-5-5', ['copilot/claude-haiku-5.5']), ('claude/sonnet', [])])
test_eq(one_each(dup, keep='copilot/claude-haiku-5.5')[0]['value'], 'copilot/claude-haiku-5.5')

An alias such as `claude/opus` and its version, such as Claude Opus 5.5, are one model. The list shows only the version, with the alias in `also`. A saved alias stays.

In [ ]:
al = [dict(value='claude/opus', name='Claude Opus', provider='claude'), dict(value='claude/claude-opus-4-1', name='Claude Opus 4.1', provider='claude'),
      dict(value='claude/claude-opus-5-5', name='Claude Opus 5.5', provider='claude'), dict(value='codex/gpt-5', name='GPT-5', provider='codex'),
      dict(value='codex/gpt-5-mini', name='GPT-5 Mini', provider='codex')]
test_eq([(r['name'], r['also']) for r in one_each(al)], [('Claude Opus 4.1', []), ('Claude Opus 5.5', ['claude/opus']), ('GPT-5', []), ('GPT-5 Mini', [])])
test_eq([r['name'] for r in one_each(al, keep='claude/opus')][:2], ['Claude Opus', 'Claude Opus 4.1'])

`Router` is the `ask` the app gives the tutor. Called as it is, it uses the default model. `for_(task)` gives the `ask` for one
task: the model the settings name for it, else the default. `for_(task, name, job)` gives the `ask` for `task` on the model `name`, on the
ramabana job `job`. The Test button of the Settings page uses the job `test:<task>`, so a test does not move the job of the task. When the default routes by task, as `RouteAsk` below does, `for_` lets it make the `ask`.
`uses(task)` gives the model id that the task runs on now. `cap(task)` gives the reply cap of the task, read on each call. `warm` starts the backend of each task in the background.

In [ ]:
#| export
AGENTS = ('build',)   # the tasks of an agent: their own default model until you pick one, and no reply cap
AGENT_MODELS = dict(build='claude-opus-5-5')   # the default model of each agent task: ramabana's `DFLT_TURN`, the model of a ramabana chat
ALL_TASKS = TASKS + tuple(BORROWS) + AGENTS   # the tasks after `extract` borrow a model and a reply cap: `settings.BORROWS`
WARM = ('guide',)   # the jobs that are not in `TASKS` and that start with the app, each on its own model or the model that it borrows

class Router:
    "The tutor's model: a default `ask`, and one model for each task that the settings name."
    def __init__(self, dflt, models=None, tokens=None):
        self.dflt = dflt
        self.models = models or (lambda: settings().get('models', {}))
        self.tokens = tokens or (lambda: settings().get('tokens', {}))
    @property
    def model(self) -> str: return getattr(self.dflt, 'model', '')
    def __call__(self, prompt, sp=''): return self.dflt(prompt, sp)
    def own(self, task:str) -> str:
        "The model saved for `task` itself, or ''."
        n = (self.models() or {}).get(task)
        return n if isinstance(n, str) else ''
    def name(self, task:str) -> str:
        "The model for `task`: its own, else the model of the task it borrows from, else '' for the default."
        return self.own(task) or (self.own(BORROWS[task]) if task in BORROWS else '')
    def named(self) -> bool:
        "Whether the owner named the default model for this run: `--model` (the model of the default `ask`) or `$ABHYASA_MODEL`."
        return bool(getattr(self.dflt, '_model', None) or os.getenv('ABHYASA_MODEL'))
    def uses(self, task:str) -> str:
        """The model id that `task` runs on now: `name(task)`, else the default model. An agent task with no model of its own runs on its own default
        (`AGENT_MODELS`), unless the owner named the default model for this run (`named`). The setting `models.default` does not change it."""
        return self.name(task) or (AGENT_MODELS[task] if task in AGENT_MODELS and not self.named() else self.model)
    def cap(self, task:str) -> int:
        "The reply cap of `task` (or of the task it borrows from): the saved value made to fit its range, else `TOKENS`."
        t = BORROWS.get(task, task)
        return fit(SETTINGS[f'tokens.{t}'], (self.tokens() or {}).get(t))
    def for_(self, task:str, name:str=None, job:str=None):
        "The `ask` for `task` on `name`, else on the model of the settings, else on the default. A routed call goes to the job `job`, else `task`. A task with no model of its own uses the job of the task it borrows from."
        n = self.name(task) if name is None else name
        job = job or (task if task not in BORROWS or self.own(task) else BORROWS[task])
        if hasattr(self.dflt, 'via'): return self.dflt.via(job, n, self.cap(task))
        return model_ask(n) if n else self.dflt
    def warm(self, jobs=None):
        "Start the backend of each job in `jobs` (default: each task, and each job of `WARM`) in a background thread, with no model call. Returns the thread, or None."
        if not hasattr(self.dflt, 'warm'): return None
        names = {j: self.name(j) for j in (TASKS + WARM if jobs is None else jobs)}
        t = threading.Thread(target=contextvars.copy_context().run, args=(self.dflt.warm, names), daemon=True)
        t.start()
        return t

def for_task(ask, task:str):
    "The `ask` for `task` when `ask` routes by task, else `ask` itself."
    return ask.for_(task) if hasattr(ask, 'for_') else ask

def hosted(ask) -> bool:
    "Whether a host app owns the model of `ask` (its default `ask` sets `hosted`): then abhyasa makes no ramabana agent of its own."
    return bool(getattr(getattr(ask, 'dflt', ask), 'hosted', False))

A router with no settings uses the default for every task. A task that names a model gets that model.

In [ ]:
dflt = Fake(dflt='default')
r = Router(dflt, models=lambda: {})
test_eq(r('x'), 'default')
test_is(r.for_('grade'), dflt)
named = Fake(dflt='named'); named.model = 'fast'
_asks['fast'] = named
r = Router(dflt, models=lambda: dict(grade='fast'))
test_eq(r.for_('grade')('x'), 'named')
test_eq(r.for_('question')('x'), 'default')
test_is(for_task(dflt, 'grade'), dflt)

`uses` names the model of a task: its own, else the model of the task it borrows from, else the default model. The builder chat (`build`) borrows from no task. It is an agent, so until you pick its model it runs on Claude Opus, the model of a ramabana chat, and not on the default model. `--model` or `$ABHYASA_MODEL` names the model of the whole run, so it changes the builder too.

In [ ]:
d = Fake(dflt='default'); d.model = 'claude/sonnet'
r = Router(d, models=lambda: dict(question='fast'))
test_eq(r.uses('talk'), 'fast')
test_eq(r.uses('build'), 'claude-opus-5-5')
r = Router(d, models=lambda: dict(build='claude/opus'))
test_eq(r.uses('build'), 'claude/opus')
assert 'build' in ALL_TASKS and 'build' not in TASKS

A model that the owner names for the run, with `--model` or `$ABHYASA_MODEL`, is the model of the builder too. Its own pick still wins.

In [ ]:
from ramabana.core import DFLT_TURN, ready_local

In [ ]:
os.environ['ABHYASA_MODEL'] = 'claude/haiku'
try:
    test_eq(Router(d).uses('build'), 'claude/sonnet')
    test_eq(Router(d, models=lambda: dict(build='claude/opus')).uses('build'), 'claude/opus')
finally: del os.environ['ABHYASA_MODEL']
test_eq(Router(d).uses('build'), 'claude-opus-5-5')
test_eq(AGENT_MODELS['build'], DFLT_TURN)   # the same model as a ramabana chat

### Every call through ramabana

ramabana is the agent layer on top of the model layer. Its `Routing` sends each job to a model. When a cloud model fails or
gives no answer, `Routing` puts it aside for five minutes and sends the job to the next route: other cloud models, then a small model
on this Mac. Its backends stay open, so a call does not pay the start time of a new `Chat`.

`ramabana_agent` makes one ramabana `Agent` for the process, on the first call. The `Agent` gets only what `Agent.oneshot` needs:

- `NullHost`: no folders, and each file operation is refused.
- `readonly=True`, no sub-agents, no extensions, no skills, no lessons: no tool can write, run code or go on the network.
- Approvals that refuse when nobody answers, and no saved rules.
- `cfg=None`: no history, plan or run log, so the `Agent` writes no files.
- A short briefing, so that it does not read skills or project files.

The turn of the `Agent` is the default model, so a task whose model fails goes to the default model first.

In [ ]:
#| export
import warnings
MAX_TOKENS = 4096   # the reply cap of a call with no task; ramabana's own cap of 1024 can cut a long reply
_ra, _ra_lock = {}, threading.RLock()

def ramabana_agent(turn:str=None):
    "The one ramabana `Agent` of this process, made on first use by ramabana's `model_agent` on the turn `dflt_model(turn)`: no folders, no tool that acts, no files."
    with _ra_lock:
        if 'agent' not in _ra:
            from ramabana.agent import model_agent
            _ra['agent'] = model_agent(dflt_model(turn), sp='You are the model of abhyasa, a tutor.')
        return _ra['agent']

Each task is a ramabana job with the same name. `_point` sets the model of the job before each call, only when the settings changed it.
A model that ramabana put aside stays aside until its time is up.

In [ ]:
#| export
def _point(a, job:str, name:str):
    "Point the ramabana job `job` at the model `name`, when it is not there yet."
    with _ra_lock:
        if a.routing.policy.get(job) == name: return
        try: a.routing.set(name, job=job)
        except Exception as e: raise RuntimeError(f'The {job} task: ramabana does not know the model {name!r} ({e}).') from e

`ramabana_oneshot` asks one question on the job of a task, through ramabana's `Agent.oneshot` with `wait=True` and `strict=True`.
With `wait`, a call waits for a backend that another call holds, on the first route and on each fallback. With `strict`, a call that no
route answers raises ramabana's `OneshotEmpty`, with the routes that ramabana put aside or the note of the route.

A failure can take some time: ramabana tries each route in turn, then the small model on this Mac. It puts a failed cloud model aside for five minutes.

A route can also raise ramabana's `OneshotTimeout` when a call goes past its deadline. `no_answer` changes each of the two errors into one
`RuntimeError` that names the task, the model and why, so each caller catches one kind of error.

`prepare` lets the caller change the text for each route, as ramabana's `Agent.oneshot` does. The guide uses it for its guard:
the guard then knows if the model of the route is on this Mac. `HOSTED` is the model that a guard takes when it does not know
the route.

In [ ]:
#| export
def no_answer(job:str, name:str, e) -> RuntimeError:
    "The `RuntimeError` for ramabana's `OneshotEmpty` or `OneshotTimeout` `e` on the job `job`: it names the task, the model `name` (else the route of `e`) and why."
    why = f'{e.route} did not answer in {e.elapsed:.0f} seconds.' if isinstance(e, TimeoutError) else e.why
    return RuntimeError(f"The {job} task got no answer from {name or getattr(e, 'route', '')} through ramabana: {why}")

HOSTED = AttrDict(name='hosted', local=False)   # the model that a guard takes when it does not know the route: one off this Mac

def ramabana_oneshot(job:str, name:str, prompt:str, sp:str='', max_tokens:int=MAX_TOKENS, prepare=None) -> str:
    "Ask `prompt` on the ramabana job `job`, pointed at the model `name`. Wait for a busy backend. Raise `no_answer` when no route answers. `prepare(prompt, sp, backend)` gives the text for each route, so a guard knows the model that gets it."
    from ramabana.runtime import OneshotEmpty, OneshotTimeout
    a = ramabana_agent(name)
    _point(a, job, name)
    try: return a.oneshot(prompt, sp, job=job, max_tokens=max_tokens, prepare=prepare, wait=True, strict=True)
    except (OneshotEmpty, OneshotTimeout) as e: raise no_answer(job, name, e) from e

`ramabana_stream` is the twin of `ramabana_oneshot` that gives the reply as the model sends it, through ramabana's
`Agent.oneshot_stream`, with the same `wait` and `strict`, and it raises the same errors. The guard (`prepare`) runs before the
stream starts. With `fallback=True`, a route that cannot stream gives its whole reply as one chunk, so the caller does not need a second path.

In [ ]:
#| export
def ramabana_stream(job:str, name:str, prompt:str, sp:str='', max_tokens:int=MAX_TOKENS, prepare=None):
    "The reply of `ramabana_oneshot`, one chunk at a time as the model sends it. A route that cannot stream gives the whole reply as one chunk. Raise `no_answer` when no route answers."
    from ramabana.runtime import OneshotEmpty, OneshotTimeout
    a = ramabana_agent(name)
    _point(a, job, name)
    try: yield from a.oneshot_stream(prompt, sp, job=job, max_tokens=max_tokens, prepare=prepare, wait=True, strict=True, fallback=True)
    except (OneshotEmpty, OneshotTimeout) as e: raise no_answer(job, name, e) from e

`RouteAsk` is the default `ask` of the web app. `via(task, name, max_tokens)` gives the `ask` of one task, and `Router.for_` uses it. A call with
no task goes to the job `ask:<model>`. `warm` starts the backend of each task, with no model call. `route_ask` gives a `RouteAsk`:
Leela imports this name.

In [ ]:
#| export
class RouteAsk:
    "The default `ask`: each call goes through the routing of ramabana, on the job of its task."
    def __init__(self, model:str=None): self._model = model
    @property
    def model(self) -> str:
        "The default model now: `model`, else `$ABHYASA_MODEL`, else the setting `models.default`, else `DFLT_MODEL`."
        return dflt_model(self._model)
    def via(self, task:str, name:str='', max_tokens:int=None, prepare=None):
        "The `ask` for `task` on the model `name`, else on the default model, with the reply cap `max_tokens`, else the default of `task`. `prepare` goes to `ramabana_oneshot`. `ask.stream(prompt, sp)` gives the reply in chunks (`ramabana_stream`)."
        name, cap = name or self.model, max_tokens or TOKENS.get(task, MAX_TOKENS)
        def ask(prompt, sp=''): return ramabana_oneshot(task, name, prompt, sp, cap, prepare)
        ask.stream = lambda prompt, sp='': ramabana_stream(task, name, prompt, sp, cap, prepare)
        ask.model = name
        return ask
    def __call__(self, prompt, sp=''): return self.via(f'ask:{self.model}')(prompt, sp)
    def warm(self, names:dict):
        "Make the `Agent` and start the backend of each task in `names` (task -> model, '' for the default). No model call."
        a = ramabana_agent(self.model)
        for task, name in names.items():
            try:
                _point(a, task, name or self.model)
                a.warm_backend(task)   # starts the backend under the lock of the `Agent`, so a call at the same time waits for it
            except Exception as e: warnings.warn(f'abhyasa could not start the model of the {task} task: {e}')

def route_ask(model:str=None) -> RouteAsk:
    "The default `ask`: a `RouteAsk` on `model`."
    return RouteAsk(model)

#### The worker agent

`ramabana_worker` makes a second ramabana `Agent`, for the gather agent of research. It is never the one `Agent` of `ramabana_agent`, so a long gather never holds a one-shot call. It runs on the default routing (`Routing(turn=dflt_model())`). Its host is the shalya `host` (default: a `NullHost`, with no folders). The caller names its tools in `tools`, then adds them with `add_tool`. ramabana's `only_tools` offers the model only those tools, and refuses a tool with another name. ramabana's memory and plan tools are held back. It has no approvals listener, so a tool that writes is refused. The caps stop a turn after `max_calls` tool calls, `max_steps` model steps, `max_tokens` tokens or `max_cost` dollars. A model such as Claude Code on a plan reports no cost. Then ramabana estimates the dollars from the tokens at the price of the model (`ramabana.agent.token_rate`), and keeps the estimate in `estimated`, so the dollar cap still holds.

In [ ]:
#| export
WORKER_SP = 'You are the source gatherer of abhyasa, a tutor.'
WORKER_CALLS, WORKER_STEPS, WORKER_DOLLARS = 300, 80, 5.   # the caps of one turn: tool calls, model steps, dollars
WORKER_TOKENS = 2_000_000   # the token cap of one turn, a backstop: the dollar cap, from the price of the model, stops a turn first

def ramabana_worker(tools:list=(), host=None, sp:str=WORKER_SP, turn:str=None, max_calls:int=WORKER_CALLS, max_steps:int=WORKER_STEPS,
                    max_cost:float=WORKER_DOLLARS, max_tokens:int=WORKER_TOKENS):
    "A new ramabana `Agent`, made by ramabana's `model_agent` on the turn `dflt_model(turn)`, with the shalya `host` (default: no folders). The model sees only the tools named in `tools`, when the caller adds them. Never the agent of `ramabana_agent`. A turn stops at `max_tokens` tokens, and at `max_cost` dollars (estimated from the tokens when the model reports no cost)."
    from ramabana.agent import model_agent
    return model_agent(dflt_model(turn), sp=sp, host=host, only_tools=list(tools), max_tool_calls=max_calls, max_steps=max_steps,
                       max_cost=max_cost, max_tokens=max_tokens)

LEAD_TOOLS = ('ls', 'grep', 'outline', 'view_file', 'notebook_cells', 'view_cell', 'web_search', 'read_url', 'delegate_search', 'set_plan', 'update_todo')   # ramabana's tools that only read, and its fan-out to sub-agents
WEB_TOOLS = ('web_search', 'read_url', 'delegate_search')   # the lead tools that reach the web: a sub-agent searches the web too
SUB_STEPS = 16   # the most model steps of one sub-agent

def ramabana_lead(read_roots=(), tools:list=(), sp:str=WORKER_SP, turn:str=None, minutes:float=15, max_calls:int=WORKER_CALLS,
                  max_steps:int=WORKER_STEPS, max_cost:float=WORKER_DOLLARS, max_tokens:int=WORKER_TOKENS, web:bool=True) -> tuple:
    """`(agent, host)`: a new ramabana `Agent` made by ramabana's `mk_agent`, with its own read and web tools (`LEAD_TOOLS`) and its
    sub-agents (`delegate_search`, at most 4 at once in the whole tree), on the turn `dflt_model(turn)`. Its sub-agents run on ramabana's
    `subagent` route. The folders `read_roots` are open for reads only; its one folder that it may write is a new empty one. It has no
    Python (`python='off'`), no vault, and no approvals listener, so each tool that writes is refused. The model sees only `LEAD_TOOLS` and
    the tools named in `tools`, when the caller adds them. The caps: `max_calls` tool calls in the whole tree, `max_steps` model steps,
    `max_tokens` tokens and `max_cost` dollars for a turn, and `minutes` for a sub-agent. With `web=False` it has no tool that reaches the web (`WEB_TOOLS`)."""
    import tempfile
    from ramabana.agent import mk_agent
    from ramabana.core import Routing
    scratch = tempfile.mkdtemp(prefix='abhyasa-lead-')
    lead = [t for t in LEAD_TOOLS if web or t not in WEB_TOOLS]
    a, host = mk_agent((scratch,), routing=Routing(turn=dflt_model(turn)), approve='ask', web=web, pii='redact', python='off',
                       read_roots=tuple(str(r) for r in read_roots), cfg=None, sp=sp, only_tools=[*lead, *tools], subagents=web,
                       subagent_writes=False, subagent_steps=SUB_STEPS, subagent_timeout=minutes * 60, subagent_depth=1, extensions=False,
                       global_skills=False, learned=False, compact=False, poll_every=0, auto_turns='off',
                       max_tool_calls=max_calls, max_steps=max_steps, max_cost=max_cost, max_tokens=max_tokens)
    host.scratch = scratch
    return a, host

The worker offers only the tool that the test names and adds. It is a new `Agent` each time, and never the shared one. The caps reach the `Agent`.

In [ ]:
def look(q:str, # what to look for
         ) -> str:
    "Look for `q`."
    return q
w = ramabana_worker(['look'], max_calls=50, max_tokens=1000)
w.add_tool(look)
test_eq([t.__name__ for t in w.tools], ['look'])
assert w is not ramabana_worker() and w is not ramabana_agent()
test_eq((w.readonly, w.subagents, w.tool_budget, w.max_cost, w.max_tokens), (True, False, 50, WORKER_DOLLARS, 1000))

The worker refuses a tool that `tools` does not name. A worker with no names offers no tool, so ramabana's own tools stay out.

In [ ]:
def write(path:str, # the file
          ) -> str:
    "Write `path`."
    return path
test_fail(lambda: w.add_tool(write), contains='not in only_tools')
test_eq(ramabana_worker().tools, [])

When the model reports no cost, ramabana estimates the dollars from the tokens with its own price table (`est_rate` is None), and
keeps them in `estimated`. A new worker has estimated nothing, and has the default token cap.

In [ ]:
w = ramabana_worker()
test_eq((w.est_rate, w.estimated, w.max_tokens), (None, 0., WORKER_TOKENS))

The lead of research is made by ramabana's `mk_agent`. It offers ramabana's read and web tools and `delegate_search`, and the tool that the test adds. It has no tool that writes a file or runs a command. The folder of the learner is open for reads only.

In [ ]:
import tempfile, shutil

In [ ]:
ro = tempfile.mkdtemp()
a, h = ramabana_lead([ro], ['look'], minutes=2)
a.add_tool(look)
names = {t.__name__ for t in a.tools}
assert {'view_file', 'read_url', 'web_search', 'delegate_search', 'look'} <= names, names
assert not names & {'run_shell', 'run_python', 'create_file', 'replace_text', 'git_commit', 'add_root'}, names
test_eq((h.read_roots, a.subagent_timeout, a.subagent_writes), ([str(Path(ro).resolve())], 120, False))
a.close(); shutil.rmtree(h.scratch, ignore_errors=True)
a, h = ramabana_lead([ro], ['look'], web=False)
assert not {t.__name__ for t in a.tools} & set(WEB_TOOLS)
a.close(); shutil.rmtree(h.scratch, ignore_errors=True)

#### Tests of the route

These tests use a ramabana `Agent` with fake backends from `ramabana.testing`, so no model runs. `using(a)` puts that `Agent` in
place of the real one for one test. CI does not install the `models` extra, so these tests run only when ramabana is installed.

In [ ]:
from contextlib import contextmanager
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

In [ ]:
from ramabana.core import resolve
from ramabana.testing import fake_agent, FakeBackend

In [ ]:
@contextmanager
def using(a):
    "Use the `Agent` `a` in place of the real one, for one test."
    _ra['agent'] = a
    try: yield a
    finally: _ra.pop('agent', None)

Each task goes to the ramabana job with the same name. The task that the settings name gets its model, the other tasks get the
default. Each call asks for the reply cap of its task, `TOKENS[task]`, so a long question with its rubric is not cut.

In [ ]:
a, be = fake_agent()
jobs, one = [], a.oneshot
a.oneshot = lambda p, sp='', job='oneshot', max_tokens=None, **kw: jobs.append((job, max_tokens)) or one(p, sp, job, max_tokens, **kw)
with using(a):
    r = Router(RouteAsk('claude/sonnet'), models=lambda: dict(grade='claude/haiku'), tokens=lambda: {})
    test_eq(r.for_('grade')('hi'), 'ONESHOT:hi')
    test_eq(r.for_('question').model, 'claude/sonnet')
    r.for_('question')('hi')
test_eq(jobs, [('grade', TOKENS['grade']), ('question', TOKENS['question'])])
test_eq((a.routing.policy['grade'], a.routing.policy['question']), ('claude/haiku', 'claude/sonnet'))

A change in the settings has an effect on the next call, with no restart. An empty choice goes back to the default.

In [ ]:
a, be = fake_agent()
ms = dict(grade='claude/haiku')
with using(a):
    r = Router(RouteAsk('claude/sonnet'), models=lambda: ms)
    r.for_('grade')('hi'); test_eq(a.routing.policy['grade'], 'claude/haiku')
    ms['grade'] = 'claude/opus'; r.for_('grade')('hi'); test_eq(a.routing.policy['grade'], 'claude/opus')
    ms['grade'] = ''; r.for_('grade')('hi'); test_eq(a.routing.policy['grade'], 'claude/sonnet')

`prepare` gets each route. Here it writes the name of the model before the prompt, so the reply shows what the route got:

In [ ]:
a, be = fake_agent()
with using(a): test_eq(RouteAsk('claude/sonnet').via('hint', prepare=lambda p, sp, b: (f'{b.spec.name} {p}', sp))('hi'), 'ONESHOT:fake hi')

`Slow` is a fake backend that takes a moment to answer. It sets `inside` when a call starts.

In [ ]:
class Slow(FakeBackend):
    "A fake backend that answers after 0.2 s and sets `inside` when a call starts."
    def __init__(self, *a, **kw): super().__init__(*a, **kw); self.inside = threading.Event()
    def _oneshot(self, prompt, sp, max_tokens): self.inside.set(); time.sleep(0.2); return f'ONESHOT:{prompt}'

Without `wait`, ramabana's `Agent.oneshot` does not wait for a busy backend: a second call at the same time gets ''.

In [ ]:
a, _ = fake_agent(); slow = Slow(); a._be_or_none = lambda job='turn': slow
with ThreadPoolExecutor(2) as ex:
    f1 = ex.submit(a.oneshot, 'one'); slow.inside.wait(); f2 = ex.submit(a.oneshot, 'two')
test_eq((f1.result(), f2.result()), ('ONESHOT:one', ''))

Through the route (`wait=True`), the second call waits for the first, and both calls get an answer.

In [ ]:
slow.inside.clear()
with using(a), ThreadPoolExecutor(2) as ex:
    ask = RouteAsk('claude/sonnet').via('question')
    f1 = ex.submit(ask, 'one'); slow.inside.wait(); f2 = ex.submit(ask, 'two')
    test_eq((f1.result(), f2.result()), ('ONESHOT:one', 'ONESHOT:two'))

When no route gives an answer, the call raises. The error names the task, the model and the note of ramabana, so `ask_json` never gets ''.

In [ ]:
class Empty(FakeBackend):
    "A fake cloud backend that answers nothing."
    def _oneshot(self, prompt, sp, max_tokens): return ''
a, _ = fake_agent(); empty = Empty(resolve('claude/haiku')); a._be_or_none = lambda job='turn': empty
with using(a):
    ask = Router(RouteAsk('claude/sonnet'), models=lambda: dict(grade='claude/haiku')).for_('grade')
    test_fail(lambda: ask('hi'), contains='grade task')
    try: ask('hi')
    except RuntimeError as e: assert 'claude/haiku' in str(e) and 'answered nothing' in str(e), e

A route that goes past its deadline raises ramabana's `OneshotTimeout`. The call raises the same `RuntimeError`, with the model and the seconds, and keeps the first error as its cause.

In [ ]:
from ramabana.runtime import OneshotTimeout

In [ ]:
a, _ = fake_agent()
def late(*a, **kw): raise OneshotTimeout('claude/haiku', 31.6, True)
a.oneshot = late
with using(a):
    try: RouteAsk('claude/haiku').via('grade')('hi'); assert False, 'no error'
    except RuntimeError as e:
        test_eq(str(e), 'The grade task got no answer from claude/haiku through ramabana: claude/haiku did not answer in 32 seconds.')
        assert isinstance(e.__cause__, OneshotTimeout)

#### Tests of the stream

`Chunks` is a fake backend that can stream: it sends its reply three letters at a time. `ask.stream` gives the chunks in order,
and their join is the reply. The guard of `prepare` sees the route before the stream starts.

In [ ]:
class Chunks(FakeBackend):
    "A fake backend that streams `ONESHOT:<prompt>` in chunks of 3 letters."
    supports_stream = True
    def _oneshot_stream(self, prompt, sp, max_tokens): return (f'ONESHOT:{prompt}'[i:i+3] for i in range(0, len(f'ONESHOT:{prompt}'), 3))
a, _ = fake_agent(); ch = Chunks(resolve('claude/haiku')); a._be_or_none = lambda job='turn': ch
with using(a): got = list(RouteAsk('claude/haiku').via('hint', prepare=lambda p, sp, b: (f'{b.spec.name} {p}', sp)).stream('hi'))
test_eq(''.join(got), 'ONESHOT:claude/haiku hi')
assert len(got) > 1

A backend that cannot stream (the default of `FakeBackend`) gives the whole reply of the blocking call as one chunk:

In [ ]:
a, be = fake_agent()
with using(a): test_eq(list(RouteAsk('claude/sonnet').via('hint').stream('hi')), ['ONESHOT:hi'])

The stream lets go of its backend at the end, also when it stops early, so the next call on that backend does not wait:

In [ ]:
a, _ = fake_agent(); a._be_or_none = lambda job='turn': ch
with using(a), ThreadPoolExecutor(1) as ex:
    list(RouteAsk('claude/haiku').via('hint').stream('one'))
    test_eq(ex.submit(RouteAsk('claude/haiku').via('hint'), 'two').result(timeout=2), 'ONESHOT:two')
    it = RouteAsk('claude/haiku').via('hint').stream('three'); next(it); it.close()   # a stream that stops early lets go too
    test_eq(ex.submit(RouteAsk('claude/haiku').via('hint'), 'four').result(timeout=2), 'ONESHOT:four')

A stream with no text raises the same error as a blocking call, and a timeout raises the same `RuntimeError`:

In [ ]:
a, _ = fake_agent(); empty = Empty(resolve('claude/haiku')); empty.supports_stream = True
empty._oneshot_stream = lambda prompt, sp, max_tokens: (c for c in ())
a._be_or_none = lambda job='turn': empty
with using(a): test_fail(lambda: list(RouteAsk('claude/haiku').via('grade').stream('hi')), contains='grade task')
def late(*a, **kw): raise OneshotTimeout('claude/haiku', 31.6, True); yield
a.oneshot_stream = late
with using(a): test_fail(lambda: list(RouteAsk('claude/haiku').via('grade').stream('hi')), contains='did not answer in 32 seconds')

The next tests need ramabana's real `Routing`, so that a failed model goes to its next route. `routed_agent` gives a fake `Agent` with
the real `Routing` on two fake models: `fake-main`, the default, and `fake-quick`. They are hosted models with a fake key, so
`Routing` takes them on any machine. `make(spec)` gives the fake backend of each model, and `bes` keeps them by name.

In [ ]:
from ramabana.core import register_model, Routing

In [ ]:
os.environ['ABHYASA_FAKE_KEY'] = 'not a real key'
for n in ('fake-main', 'fake-quick'): register_model(n, f'fakevendor/{n}', runtime='remote', api_key_env='ABHYASA_FAKE_KEY')

def routed_agent(make):
    "A fake `Agent` on the real `Routing`, with the turn on `fake-main`. Returns the `Agent` and the backends by model name."
    a, _ = fake_agent(routing=Routing(turn='fake-main'))
    del a._be_or_none   # the real one, which asks `Routing`
    bes = {}
    def be(job='turn'):
        s = a.routing.spec(job)
        if s.name not in bes: bes[s.name] = make(s)
        return bes[s.name]
    a._be = be
    return a, bes

A fallback waits for the backend that it goes to. Here `fake-quick` answers nothing, so the `grade` call goes to `fake-main`.
A `question` call on `fake-main` starts while that fallback runs. It waits for the backend, and both calls get an answer.

In [ ]:
a, bes = routed_agent(lambda s: Slow(s) if s.name != 'fake-quick' else Empty(s))
with using(a), ThreadPoolExecutor(2) as ex:
    g = ex.submit(Router(RouteAsk('fake-main'), models=lambda: dict(grade='fake-quick'), tokens=lambda: {}).for_('grade'), 'g')
    while 'fake-main' not in bes: time.sleep(0.01)
    bes['fake-main'].inside.wait()
    q = ex.submit(RouteAsk('fake-main').via('question'), 'q')
    test_eq((g.result(), q.result()), ('ONESHOT:g', 'ONESHOT:q'))

When no route answers, each route is tried one time. The last route is the small model on this Mac, `gemma-e4b`.

In [ ]:
calls = Counter()
class Counted(Empty):
    "A fake backend that answers nothing and counts its calls."
    def _oneshot(self, prompt, sp, max_tokens): calls[self.spec.name] += 1; return ''
a, bes = routed_agent(Counted)
with using(a): test_fail(lambda: RouteAsk('fake-main').via('grade', 'fake-quick')('hi'), contains='grade task')
if (loc := ready_local()): test_eq(calls[loc], 1)   # a local model only when its weights are on this machine
test_eq(set(calls.values()), {1})

`warm` asks ramabana for the backend of each task in the background, and sends nothing to the model. It gives the thread.
The jobs of `WARM` start too: the `guide` job, on the model of the screen guide (the model for hints when it has none). `warm(['guide'])` starts only the guide.

In [ ]:
a, be = fake_agent(); seen = []
a._be_or_none = lambda job='turn': seen.append(job) or be
with using(a):
    Router(RouteAsk('claude/sonnet'), models=lambda: {}).warm().join()
test_eq(seen, [*TASKS, 'guide'])
test_eq(be.sent, [])
seen.clear()
with using(a): Router(RouteAsk('claude/sonnet'), models=lambda: dict(hint='claude/haiku')).warm(['guide']).join()
test_eq((seen, a.routing.policy['guide']), (['guide'], 'claude/haiku'))
with using(a): Router(RouteAsk('claude/sonnet'), models=lambda: dict(hint='claude/haiku', guide='claude/opus')).warm(['guide']).join()
test_eq(a.routing.policy['guide'], 'claude/opus')   # a model of its own wins

`route_ask` gives a `RouteAsk`.

In [ ]:
assert isinstance(route_ask('claude/haiku'), RouteAsk) and route_ask('claude/haiku').model == 'claude/haiku'

#### The reply cap of each task

A reply cap comes from the settings table (`tokens.<task>`), from 1 to `TOKENS_MAX`. A cap that you typed in the file as text, such as `"900"`, works too.

In [ ]:
from abhyasa.learner import update_settings, val

In [ ]:
update_settings('tokens', hint=' 900 ')
test_eq(val('tokens.hint'), 900)
update_settings('tokens', hint=str(TOKENS_MAX + 1))
test_eq(val('tokens.hint'), TOKENS_MAX)   # clamped to the top
update_settings('tokens', hint=None)

`Router.cap` gives the saved cap of a task, else the default of the task. A bad value in the settings file gives the default, so a
call never fails because of it.

In [ ]:
caps = dict(hint=300)
r = Router(Fake(), models=lambda: {}, tokens=lambda: caps)
test_eq(r.cap('hint'), 300)
test_eq(r.cap('grade'), TOKENS['grade'])
caps['hint'] = 'lots'; test_eq(r.cap('hint'), TOKENS['hint'])
caps['hint'] = 99999; test_eq(r.cap('hint'), TOKENS_MAX)          # out of range: the most

A task after `extract` (`plan`, `pick`, `note`, `profile`) borrows the model, the reply cap and the job of an older task until it has a model of its own.

In [ ]:
class Via:
    model = 'base'
    def __call__(self, p, sp=''): return ''
    def via(self, job, name='', cap=None): return (job, name, cap)
ms = dict(extract='x/extract', hint='x/hint')
r = Router(Via(), models=lambda: ms, tokens=lambda: {})
test_eq((r.for_('plan'), r.for_('note')), (('extract', 'x/extract', 4096), ('hint', 'x/hint', 512)))
ms['plan'] = 'x/plan'
test_eq(r.for_('plan'), ('plan', 'x/plan', 4096))

The cap reaches `Agent.oneshot`, and a change has an effect on the next call.

In [ ]:
a, be = fake_agent()
jobs, one = [], a.oneshot
a.oneshot = lambda p, sp='', job='oneshot', max_tokens=None, **kw: jobs.append((job, max_tokens)) or one(p, sp, job, max_tokens, **kw)
caps = dict(grade=777)
with using(a):
    r = Router(RouteAsk('claude/sonnet'), models=lambda: {}, tokens=lambda: caps)
    r.for_('grade')('hi'); caps['grade'] = ''; r.for_('grade')('hi')
test_eq(jobs, [('grade', 777), ('grade', TOKENS['grade'])])

The Test button of the Settings page asks on its own job, `test:<task>`, with the cap of the task. The job of the task stays where
the settings put it.

In [ ]:
a, be = fake_agent()
jobs, one = [], a.oneshot
a.oneshot = lambda p, sp='', job='oneshot', max_tokens=None, **kw: jobs.append((job, max_tokens)) or one(p, sp, job, max_tokens, **kw)
with using(a):
    r = Router(RouteAsk('claude/sonnet'), models=lambda: dict(grade='claude/haiku'), tokens=lambda: {})
    r.for_('grade')('hi')
    r.for_('grade', 'claude/opus', job='test:grade')('hi')
test_eq(jobs, [('grade', TOKENS['grade']), ('test:grade', TOKENS['grade'])])
test_eq((a.routing.policy['grade'], a.routing.policy['test:grade']), ('claude/haiku', 'claude/opus'))

## The guard

> Text that leaves the Mac passes the PII guard first.

Text that leaves the Mac passes ramabana's guard (`ramabana.guard`) first. `guard(s, model)` is for a model call: the page
context and the question, and the prompt of a vision call. It is `outgoing` for the model of the route. It always masks API
keys, tokens and private keys as `[SECRET]`. For personal data it follows ramabana's setting `cli.pii`, but only for a hosted
model: `redact` masks it, `refuse` withholds the text (`Withheld`), and `off` lets it through. A model on this Mac gets the
personal data. When the route is not known, `guard` takes a hosted model (`HOSTED`).

`safe_query(q)` is for a web query. It is ramabana's `query`: a query with a secret or with personal data does not go, with any
setting.

The guard reads text only. A figure image goes to the vision model as it is: no text guard can read pixels.

In [ ]:
#| export
class Withheld(Exception):
    "The guard withheld the text: it holds personal data and ramabana's `cli.pii` is `refuse`, or it is a web query with a secret or personal data."

def guard(s, model=None, mode:str=None) -> str:
    "`s` as it may go to `model` (a ramabana `ModelSpec`; default `HOSTED`), through ramabana's `outgoing`, with the PII policy `mode` (default `cli.pii`)."
    s = str(s or '')
    if not s.strip(): return s
    from ramabana.guard import OutgoingRefused, outgoing
    try: return outgoing(s, model or HOSTED, mode)
    except OutgoingRefused as e: raise Withheld(str(e)) from e

def safe_query(q:str) -> str:
    "The web query `q` as it is, through ramabana's `query`; `Withheld` when it holds a secret or personal data."
    from ramabana.guard import OutgoingRefused, query
    try: return query(q)
    except OutgoingRefused as e: raise Withheld(str(e)) from e

def guarded(p, sp, b) -> tuple:
    "The `prepare` of a routed call: the prompt `p` and the system prompt `sp` through `guard` for the model of the backend `b`."
    return guard(p, b.spec), guard(sp, b.spec)

A key never gets through. Plain text stays as it is:

In [ ]:
test_eq(guard('key sk-test1234567890abcdefXYZ here'), 'key [SECRET] here')
test_eq(guard(r'The share \bar\alpha_t is left.'), r'The share \bar\alpha_t is left.')

With `redact`, a hosted model gets `[EMAIL]`, and a model on this Mac gets the address. A key is masked for both:

In [ ]:
local = AttrDict(name='mlx/gemma', local=True)
test_eq(guard('mail ada@example.com', mode='redact'), 'mail [EMAIL]')
test_eq(guard('mail ada@example.com sk-test1234567890abcdefXYZ', local, 'redact'), 'mail ada@example.com [SECRET]')

With `refuse`, text with personal data for a hosted model is withheld. A web query with a secret or personal data never goes:

In [ ]:
test_fail(lambda: guard('mail ada@example.com', mode='refuse'), exc=Withheld)
test_fail(lambda: safe_query('alpha bar sk-test1234567890abcdefXYZ'), exc=Withheld)
test_fail(lambda: safe_query('alpha bar ada@example.com'), exc=Withheld)
test_eq(safe_query('alpha bar diffusion'), 'alpha bar diffusion')

## Speech

Speech to text on this machine, for think-aloud answers, and a natural voice that reads the guide aloud.

This module turns a spoken answer into text. With the local engines, the audio does not go off the machine.
The web app records while you talk. It cuts the audio into segments at your pauses and sends each segment here.
This module transcribes the segments one at a time, in order. The text goes into the think-aloud box while you continue to talk.

The module also speaks. `speak` turns one line of the screen guide into WAV audio, with the start time of each word, so the board can light each word as it is said.

These are the imports of the module.

### Engines

There are two engines. `mlx` runs on this machine. `openai` sends the audio to OpenAI.

- `mlx` runs any speech to text model that mlx-audio can load. The default is Qwen3-ASR 0.6B (8 bit, 1.0 GB).
  Whisper large-v3-turbo is the second model on the Settings page, and the search finds the others.
  Install mlx-audio with `pip install abhyasa[whisper]`.
- `openai` uses the OpenAI transcription API. It is off until you set a key. It sends the audio off the machine.

Each engine gets a hint: the name of the concept and its terms. Qwen3-ASR reads it as hot words, Whisper as its initial prompt,
and OpenAI as its prompt.

The table gives the numbers on this machine (an M-series Mac), with each model warm.
"Clip A" is the English test clip of the model layer (`rishi/nbs/speech.wav`).
"Clip B" is a clip in a cloned voice with Nyāya terms in IAST and technical terms.
"Terms" is the fraction of the 9 key terms in clip B that the engine wrote correctly.
"5 s" is the time for one 5-second segment.

| model | load | 5 s | clip A | clip B, no hint / hint | terms, no hint / hint | Sanskrit CER | memory |
|---|---|---|---|---|---|---|---|
| `mlx-community/Qwen3-ASR-0.6B-8bit` | 3.8 s | 0.14 s | 6% | 13% / 0% | 5 / 9 | 16% | 2.0 GB |
| `mlx-community/whisper-large-v3-turbo-asr-fp16` | 4.8 s | 0.68 s | 6% | 4% / 0% | 8 / 9 | 10% | 2.3 GB |
| `mlx-community/Qwen3-ASR-1.7B-8bit` | 3.6 s | 0.25 s | 6% | 6% / 0% | 7 / 9 | 14% | 3.6 GB |
| `mlx-community/parakeet-tdt-0.6b-v3` | 4.9 s | 0.15 s | 6% | 4% / 4% | 7 / 7 | Latin script | 3.2 GB |
| `mlx-community/Voxtral-Mini-4B-Realtime-2602-4bit` | 3.7 s | 2.21 s | 6% | 6% / 6% | 7 / 7 | 22% | 5.5 GB |
| `UsefulSensors/moonshine-base` | 3.9 s | 0.07 s | 100% | 8% / 8% | 6 / 6 | Latin script | 0.8 GB |
| `Mediform/canary-1b-v2-mlx-q8` | 4.2 s | 0.08 s | 6% | 4% / 4% | 8 / 8 | Latin script | 1.4 GB |

- "load" is a cold start of a cached model in a new process, with the Python imports. The first download takes longer.
- "memory" is the MLX peak.
- "Sanskrit CER" is the character error rate on a 9-second Nyāya sūtra (from vachaka), after the vachaka normalization.
- Moonshine gave no text for the 24-second clip A. It is good only for short segments.
- For Sanskrit speech, use Qwen3-ASR or Whisper with the hint, or vachaka.

Qwen3-ASR 0.6B is the default. With the hint, it has no errors on clip B and it writes all 9 terms. One 5-second segment takes 0.14 s.
Before 0.0.16, the default was Gemma 4 E4B on LiteRT. It had the same score, but it was 5 times slower, and its model was 3.7 GB.

`DFLT_STT` is the default model of the `mlx` engine. The other constants are the named models.
`ALIASES` keeps the old engine names: `whisper` and `parakeet` are the `mlx` engine with a fixed model. `gemma` was the default
engine before 0.0.16, so it becomes the `mlx` engine with the default model. `POCKET` is the Hugging Face repo of the local voice (see "A natural voice" below), and `POCKET_FILES` are the files of its
English model and of its default voice. The repo has many languages, so abhyasa gets only these files. `VOICE_FILES` gives the files of the local voice.

In [ ]:
#| export
DFLT_STT = 'mlx-community/Qwen3-ASR-0.6B-8bit'
DFLT_WHISPER = 'mlx-community/whisper-large-v3-turbo-asr-fp16'
DFLT_PARAKEET = 'mlx-community/parakeet-tdt-0.6b-v3'
DFLT_OPENAI = 'gpt-4o-mini-transcribe'
DFLT_MODELS = dict(mlx=DFLT_STT, openai=DFLT_OPENAI)
ALIASES = dict(gemma=('mlx', DFLT_STT), whisper=('mlx', DFLT_WHISPER), parakeet=('mlx', DFLT_PARAKEET))
POCKET = 'kyutai/pocket-tts-without-voice-cloning'
POCKET_FILES = ('languages/english/model.safetensors', 'languages/english/tokenizer.json', 'languages/english/embeddings/caro_davy.safetensors')
VOICE_FILES = {POCKET: POCKET_FILES}

### Settings

The settings are in `settings.json` in `cfg_dir()`. The `settings` module keeps the table of values and the file; this module uses its `stt` section.
The web app changes the settings, and the next segment uses the change. You do not have to restart.

In [ ]:
#| export
from abhyasa.learner import cfg_dir, is_hosted, DFLT_SETTINGS, settings, save_settings, update_settings, val

Without a file, you get the defaults. A change to one key keeps the other keys.

In [ ]:
test_eq({k: settings()['stt'][k] for k in ('engine', 'model', 'segment')}, dict(engine='mlx', model='', segment=True))
update_settings('stt', segment=False)
test_eq(val('stt.segment'), False)
update_settings('stt', segment=True)

`current` gives the engine and model in use. `$ABHYASA_STT` has priority over the settings. It can be an engine (`mlx`),
an old name (`whisper`, `gemma`), or an engine and a model (`mlx:mlx-community/whisper-large-v3-turbo-asr-fp16`).
An old settings file with `gemma` and a LiteRT model gets the default model. Nothing writes the file again.

In [ ]:
#| export
def resolve(engine:str, model:str='') -> tuple:
    "The engine and model for a name: aliases become `mlx`, `engine:model` splits, an empty model gets the default."
    if ':' in engine: engine, model = engine.split(':', 1)
    if engine == 'gemma': model = ''   # its model was a LiteRT chat model, which the `mlx` engine cannot load
    if engine in ALIASES: engine, model = ALIASES[engine][0], model or ALIASES[engine][1]
    if engine not in DFLT_MODELS: raise ValueError(f'unknown speech engine {engine!r}')
    return engine, model or DFLT_MODELS[engine]

def current() -> tuple:
    "The engine and model in use: `$ABHYASA_STT`, else the settings."
    if (e := os.getenv('ABHYASA_STT')): return resolve(e)
    s = settings()['stt']
    return resolve(s['engine'], s['model'])

The settings give Qwen3-ASR by default. An old `gemma` setting gives the same, also with its old model. The environment variable has priority.

In [ ]:
test_eq(current(), ('mlx', DFLT_STT))
test_eq(resolve('gemma', 'litert-community/gemma-4-E4B-it-litert-lm'), ('mlx', DFLT_STT))
test_eq(resolve('parakeet'), ('mlx', DFLT_PARAKEET))
test_eq(resolve('mlx:mlx-community/whisper-tiny-asr-fp16'), ('mlx', 'mlx-community/whisper-tiny-asr-fp16'))
test_fail(lambda: resolve('espeak'), contains='unknown')
os.environ['ABHYASA_STT'] = 'whisper'
test_eq(current(), ('mlx', DFLT_WHISPER))
os.environ['ABHYASA_STT'] = 'gemma'
test_eq(current(), ('mlx', DFLT_STT))
del os.environ['ABHYASA_STT']

### One model in memory

`resident` keeps one model loaded in each slot: `stt` for speech to text, `tts` for the speaking voice. When the app asks for a different model
in a slot, it frees the old model of that slot first. Two large models for the same job in memory at the same time can make the Mac slow.

In [ ]:
#| export
_res, _lock = AttrDict(key=None, obj=None, close=None), threading.RLock()
_slots = dict(stt=(_res, _lock), tts=(AttrDict(key=None, obj=None, close=None), threading.RLock()))   # each slot has its own lock, so a load of one does not wait for the other

def unload(slot:str='stt'):
    "Free the loaded model of `slot`: `stt` (speech to text) or `tts` (the speaking voice)."
    r, lock = _slots[slot]
    with lock:
        if r.close and r.obj is not None: r.close(r.obj)
        r.update(key=None, obj=None, close=None)
        gc.collect()
        try:
            import mlx.core as mx; mx.clear_cache()
        except ImportError: pass

def resident(key, load, close=None, slot:str='stt'):
    "The loaded model of `slot` for `key`: when `key` is new, free the old model of the slot, then `load()` the new one."
    r, lock = _slots[slot]
    with lock:
        if r.key != key:
            unload(slot)
            r.obj = load()
            r.key, r.close = key, close
        return r.obj

A new key frees the old model. The same key does not load again. A model in the `tts` slot does not free the model in the `stt` slot.

In [ ]:
freed = []
resident('a', lambda: 'A', freed.append)
test_eq(resident('a', lambda: 'other'), 'A')
test_eq(resident('b', lambda: 'B'), 'B')
test_eq(freed, ['A'])
test_eq(resident('k', lambda: 'K', slot='tts'), 'K')
test_eq((resident('b', lambda: 'other'), freed), ('B', ['A']))
unload(); unload('tts')

Each engine is a function that takes a WAV path, a hint and a model, and returns text.

The `mlx` engine gives the hint to a model that accepts one: Whisper as `initial_prompt`, Qwen3-ASR as `hotwords`.

In [ ]:
#| export
def _hint_kw(m, hint:str) -> dict:
    "The `generate` argument that carries the hint, for models that accept one."
    ps = inspect.signature(m.generate).parameters
    if not hint: return {}
    if 'initial_prompt' in ps: return dict(initial_prompt=hint)
    if 'hotwords' in ps: return dict(hotwords=[o.strip() for o in hint.split(',') if o.strip()])
    return {}

def _mlx(path, hint='', model=DFLT_STT):
    from mlx_audio.stt.utils import load
    m = resident(('mlx', model), lambda: load(model))
    return m.generate(str(path), **_hint_kw(m, hint)).text

A Whisper-like model gets `initial_prompt`. A Qwen3-like model gets a list of hot words. A model without either gets nothing.

In [ ]:
class _W:
    def generate(self, audio, initial_prompt=None): pass
class _Q:
    def generate(self, audio, hotwords=None): pass
test_eq(_hint_kw(_W(), 'Nyāya, hetu'), dict(initial_prompt='Nyāya, hetu'))
test_eq(_hint_kw(_Q(), 'Nyāya, hetu'), dict(hotwords=['Nyāya', 'hetu']))
test_eq(_hint_kw(_W(), ''), {})

### OpenAI

The `openai` engine posts the WAV to the OpenAI transcription API, with the hint as its `prompt`.
`http_post` is `httpx.post` unless a test puts a fake in its place.

In [ ]:
#| export
OPENAI_URL = 'https://api.openai.com/v1/audio/transcriptions'
http_post = None

def _openai(path, hint='', model=DFLT_OPENAI):
    key, _ = openai_key()
    if not key: raise RuntimeError('There is no OpenAI key. Set one on the Settings page.')
    post = http_post or __import__('httpx').post
    data = dict(model=model, response_format='json', **(dict(prompt=hint) if hint else {}))
    r = post(OPENAI_URL, headers={'Authorization': f'Bearer {key}'}, data=data,
             files=dict(file=('clip.wav', Path(path).read_bytes(), 'audio/wav')), timeout=60)
    r.raise_for_status()
    return r.json()['text']

ENGINES = dict(mlx=_mlx, openai=_openai)

The key comes from the first of these places that has `OPENAI_API_KEY`: the environment, the abhyasa `keys.env`, then the ramabana `keys.env`. ramabana says where its file is (`$RAMABANA_CONFIG`, else `~/.config/ramabana/keys.env`).
abhyasa reads both files with ramabana's `read_key`. It refuses a file that other users can read, it runs no `!command` value, and it sets nothing in the environment. abhyasa only reads the ramabana file. `set_openai_key` writes the abhyasa file with `save_key` (atomic, mode 0600).
`masked` shows only the last 4 characters.

In [ ]:
#| export
def openai_key() -> tuple:
    "The OpenAI key and where it is from; `('', '')` when there is none."
    if (k := os.getenv('OPENAI_API_KEY')): return k, 'environment'
    if is_hosted(): return '', ''   # a host owns the key: it comes from its environment, never from a file
    from ramabana.setup import keys_path, read_key
    for p in (cfg_dir()/'keys.env', keys_path()):
        if (k := read_key('OPENAI_API_KEY', p)): return k, str(p)
    return '', ''

def set_openai_key(key:str=''):
    "Save `key` in abhyasa's `keys.env` (mode 0600); an empty key clears it. A key with other than letters, digits, `-` and `_` raises `ValueError`."
    if key.strip() and not re.fullmatch(r'[A-Za-z0-9_-]{8,300}', key.strip()): raise ValueError('An API key has only letters, digits, - and _.')
    from ramabana.setup import save_key
    old = os.environ.get('OPENAI_API_KEY')
    save_key('OPENAI_API_KEY', key.strip(), cfg_dir()/'keys.env')
    if old is None: os.environ.pop('OPENAI_API_KEY', None)   # `save_key` also sets the environment: only a file holds this key
    else: os.environ['OPENAI_API_KEY'] = old

def masked(key:str) -> str: return f'…{key[-4:]}' if key else ''

A key in the abhyasa file comes back with its source, and the file has mode 0600. When the abhyasa file has no key, abhyasa reads the ramabana file. This test points ramabana to a temp folder.

In [ ]:
from ramabana.setup import keys_path

In [ ]:
os.environ['RAMABANA_CONFIG'] = str(Path(tempfile.mkdtemp())/'config.toml')
test_eq(openai_key(), ('', ''))
keys_path().write_text('export OPENAI_API_KEY="sk-ramabana-0000"\n'); keys_path().chmod(0o600)
test_eq(openai_key(), ('sk-ramabana-0000', str(keys_path())))
test_eq(os.getenv('OPENAI_API_KEY'), None)
set_openai_key('sk-test-1234')
test_eq(openai_key(), ('sk-test-1234', str(cfg_dir()/'keys.env')))
test_eq(oct((cfg_dir()/'keys.env').stat().st_mode & 0o777), '0o600')
test_eq(masked('sk-test-1234'), '…1234')

A `!command` value in a key file is not run, so it gives no key:

In [ ]:
set_openai_key('')
keys_path().write_text("OPENAI_API_KEY='!echo sk-from-a-command'\n"); keys_path().chmod(0o600)
test_eq(openai_key(), ('', ''))
set_openai_key('sk-test-1234')

Hosted, the key comes from the environment only: no `keys.env` of abhyasa and none of ramabana is read.

In [ ]:
from abhyasa.learner import use_config
with use_config(tempfile.mkdtemp()):
    test_eq(openai_key(), ('', ''))
    os.environ['OPENAI_API_KEY'] = 'sk-env-9999'
    try: test_eq(openai_key(), ('sk-env-9999', 'environment'))
    finally: del os.environ['OPENAI_API_KEY']
test_eq(openai_key()[0], 'sk-test-1234')

The engine sends the model, the hint and the WAV, with the key in the header. This test uses a fake HTTP client.

In [ ]:
class _R:
    def raise_for_status(self): pass
    def json(self): return dict(text='hetu and sādhya')
sent = {}
def fake_post(url, **kw): sent.update(url=url, **kw); return _R()
http_post = fake_post
p = Path(tempfile.mkdtemp())/'a.wav'; p.write_bytes(b'RIFF')
test_eq(_openai(p, 'hetu'), 'hetu and sādhya')
test_eq(sent['data'], dict(model=DFLT_OPENAI, response_format='json', prompt='hetu'))
test_eq(sent['headers']['Authorization'], 'Bearer sk-test-1234')
set_openai_key('')
keys_path().unlink()
test_fail(lambda: _openai(p), contains='no OpenAI key')
http_post = None

A key with a newline or a space can add a second line to the file. Thus `set_openai_key` refuses it.

In [ ]:
test_fail(lambda: set_openai_key('sk-a\nOTHER=1'), contains='only letters')
test_fail(lambda: set_openai_key('sk a b c d e'), contains='only letters')

### Transcribe

`transcribe` accepts a path or bytes. It writes bytes to a temporary file and removes the file after use.
Without an engine, it uses `current()`, so a change on the Settings page applies to the next segment.

In [ ]:
#| export
def transcribe(audio, engine:str=None, hint:str='', model:str=None) -> str:
    "Text for a WAV file or WAV bytes. `hint` lists words that the speaker can use, such as the name of the concept."
    engine, model = resolve(engine, model or '') if engine else current()
    eng = partial(ENGINES[engine], model=model)
    if isinstance(audio, (bytes, bytearray)):
        with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as f: f.write(audio)
        try: return eng(f.name, hint).strip()
        finally: os.unlink(f.name)
    return eng(audio, hint).strip()

`wav_seconds` gives the length of a clip in seconds. `silence` makes a clip of silence. The app uses this clip to warm a model.

In [ ]:
#| export
def wav_seconds(audio:bytes) -> float:
    "The length of WAV bytes, in seconds; 0 when they are not WAV."
    import io
    try:
        with wave.open(io.BytesIO(audio)) as w: return w.getnframes() / w.getframerate()
    except Exception: return 0.

def silence(secs:float=.5) -> bytes:
    "WAV bytes of silence: 16 kHz, mono, 16 bit."
    import io
    b = io.BytesIO()
    with wave.open(b, 'wb') as w: w.setnchannels(1); w.setsampwidth(2); w.setframerate(16000); w.writeframes(b'\0\0' * int(16000 * secs))
    return b.getvalue()

Half a second of silence is 0.5 seconds. Bytes that are not WAV give 0.

In [ ]:
test_eq(wav_seconds(silence()), .5)
test_eq(wav_seconds(b'nope'), 0.)

### Models

`KNOWN` lists the models that the Settings page shows first: the two local models that we measured (see the table above), and two OpenAI models.
`cached` tells if a model is in the Hugging Face cache. It does not use the network. The repo of the local voice (`POCKET`, below) has
many languages, so for that repo `cached` looks for its English model only.

In [ ]:
#| export
KNOWN = [
    dict(engine='mlx', model=DFLT_STT, bench={'seg5': 0.14, 'wer_a': 0.057, 'wer_b': 0.0, 'terms': 9, 'gb': 1.0}, label='Qwen3-ASR 0.6B (8 bit)', note='the default; reads the hint as hot words'),
    dict(engine='mlx', model=DFLT_WHISPER, bench={'seg5': 0.68, 'wer_a': 0.057, 'wer_b': 0.0, 'terms': 9, 'gb': 1.6}, label='Whisper large-v3-turbo', note='reads the hint as its first words'),
    dict(engine='openai', model=DFLT_OPENAI, label='OpenAI gpt-4o-mini-transcribe', note='sends your audio off this Mac; needs a key'),
    dict(engine='openai', model='gpt-4o-transcribe', label='OpenAI gpt-4o-transcribe', note='sends your audio off this Mac; needs a key'),
]

def cached(model:str) -> bool:
    "True when `model` is in the Hugging Face cache; for the local voice, when its English model is there."
    from huggingface_hub.constants import HF_HUB_CACHE
    d = Path(HF_HUB_CACHE)/f"models--{model.replace('/', '--')}"/'snapshots'
    need = VOICE_FILES.get(model, ('',))[0]
    return d.is_dir() and any((s/need).exists() for s in d.iterdir())

def cache_bytes(model:str) -> int:
    "The bytes of `model` in the Hugging Face cache so far, with the files that are still downloading."
    from huggingface_hub.constants import HF_HUB_CACHE
    d = Path(HF_HUB_CACHE)/f"models--{model.replace('/', '--')}"/'blobs'
    return sum(f.stat().st_size for f in d.iterdir()) if d.is_dir() else 0

`search_models` looks up mlx-community speech to text models on Hugging Face, with their size. It drops text to speech, aligners
and diarization models, which the `mlx` engine cannot use. `model_size` gives the size of a download: for the local voice, only its English files.

In [ ]:
#| export
STT_WORDS = ('whisper', 'parakeet', 'asr', 'voxtral', 'canary', 'moonshine', 'granite', 'sensevoice', 'nemotron', 'stt', 'speech')
NOT_STT = ('tts', 'aligner', 'diariz', 'music', 'vad', 'speech-tokenizer')

def is_stt(model:str) -> bool:
    "True when the repo name looks like a speech to text model."
    m = model.lower()
    return any(w in m for w in STT_WORDS) and not any(w in m for w in NOT_STT)

def model_size(model:str, api=None) -> int:
    "The bytes that a download of `model` gets from Hugging Face."
    from huggingface_hub import HfApi
    fs = (api or HfApi()).model_info(model, files_metadata=True).siblings
    return sum(s.size or 0 for s in fs if model not in VOICE_FILES or s.rfilename in VOICE_FILES[model])

def search_models(q:str='', limit:int=15, api=None) -> list:
    "mlx-community speech to text models that match `q`, most downloaded first, with their size."
    from huggingface_hub import HfApi
    api = api or HfApi()
    ms = [m for m in api.list_models(author='mlx-community', search=q or None, sort='downloads', limit=400) if is_stt(m.id)][:limit]
    with ThreadPoolExecutor(8) as ex: sizes = list(ex.map(lambda m: model_size(m.id, api), ms))
    return [dict(model=m.id, downloads=m.downloads or 0, size=s) for m, s in zip(ms, sizes)]

The filter keeps speech to text and drops the others.

In [ ]:
test_eq([is_stt(m) for m in ('mlx-community/Qwen3-ASR-0.6B-8bit', 'mlx-community/whisper-tiny-asr-fp16', 'mlx-community/Qwen3-TTS-12Hz-1.7B-Base-bf16', 'mlx-community/silero-vad', POCKET)],
        [True, True, False, False, False])

`search_models` gives only the speech to text models, and `model_size` counts only the English files of the local voice.
This test uses a fake Hugging Face API, so it does not use the network.

In [ ]:
class _Api:
    def list_models(self, **kw): return [AttrDict(id=i, downloads=n) for i, n in (('mlx-community/csm-1b', 9), ('mlx-community/whisper-tiny-asr-fp16', 8))]
    def model_info(self, m, files_metadata=True): return AttrDict(siblings=[AttrDict(rfilename=f, size=10) for f in (*POCKET_FILES, 'languages/french/model.safetensors')])
test_eq(search_models('', api=_Api()), [dict(model='mlx-community/whisper-tiny-asr-fp16', downloads=8, size=40)])
test_eq(model_size(POCKET, _Api()), 30)

`fetch` downloads a model and notes it in the settings. The local voice gets only its English model and its default voice: it loads them,
so they are in memory for the first line. `remove` deletes a model from the cache, but only a model that `fetch` downloaded.
Other entries in the cache belong to other programs.

In [ ]:
#| export
def fetch(model:str):
    "Download `model` into the Hugging Face cache; note it when it was not there before."
    from huggingface_hub import snapshot_download
    new = not cached(model)
    if model == POCKET: _pocket(model, OUT_DFLT['pocket'][1])
    else: snapshot_download(model)
    if new:
        d = settings(); d['downloaded'] = sorted({*d['downloaded'], model}); save_settings(d)

def remove(model:str):
    "Delete `model` from the Hugging Face cache. Only a model that `fetch` downloaded."
    d = settings()
    if model not in d['downloaded']: raise ValueError(f'{model} was not downloaded by abhyasa, so it stays')
    for slot, (r, _) in _slots.items():
        if r.key and r.key[1] == model: unload(slot)
    from huggingface_hub import scan_cache_dir
    info = scan_cache_dir()
    revs = [r.commit_hash for repo in info.repos if repo.repo_id == model for r in repo.revisions]
    if revs: info.delete_revisions(*revs).execute()
    d['downloaded'].remove(model); save_settings(d)

`remove` refuses a model that abhyasa did not download.

In [ ]:
test_fail(lambda: remove(DFLT_WHISPER), contains='stays')

### A natural voice

The board of the screen guide says one line for each step. The browser can say it with its own voice (`speechSynthesis`), but that voice sounds
like a machine. `speak` gives a natural voice. There are two engines:

- `pocket` runs Kyutai's Pocket TTS on this Mac, with the pocket-tts-mlx package. The download is 225 MB. It has 21 English voices, and it makes
  a line of 4 seconds in less than 1 second. Install it with `pip install abhyasa[voice]`.
- `openai` sends the line to the OpenAI speech API. It is off until you set a key. Each line passes ramabana's guard first.

`browser` is the third choice of the setting `voice.out`. Then the page speaks, and this module makes no audio.
The default is `pocket`. Until its model is on this Mac, the browser speaks: `Speaker` refuses the line with the word "browser", and the page uses its own voice.

Licences: the pocket-tts-mlx code is MIT. The Pocket TTS weights (`kyutai/pocket-tts-without-voice-cloning`, by Kyutai) are CC-BY-4.0.

These are the defaults. `POCKET_VOICES` are the 21 English voices of Pocket TTS. `caro_davy` is first: the owner chose it by ear.

Not each OpenAI voice works with each model. `VOICES_OF` gives the voices of each speech model, from OpenAI's guide
(https://developers.openai.com/api/docs/guides/text-to-speech, read on 2 Nov 2026): `gpt-4o-mini-tts` has all 13 voices, and `tts-1` and
`tts-1-hd` have a smaller set of 9. Thus `ballad`, `verse`, `marin` and `cedar` work only with `gpt-4o-mini-tts`. The first voice of each
model is its default. Change this table, and only this table, when OpenAI changes its voices.

In [ ]:
#| export
POCKET_VOICES = ('caro_davy', 'alba', 'anna', 'azelma', 'bill_boerst', 'charles', 'cosette', 'eponine', 'eve', 'fantine', 'george',
                 'jane', 'javert', 'jean', 'marius', 'mary', 'michael', 'paul', 'peter_yearsley', 'stuart_bell', 'vera')
DFLT_SPEECH = 'gpt-4o-mini-tts'
OUT_ENGINES = ('pocket', 'openai', 'browser')
OPENAI_VOICES = ('marin', 'cedar', 'coral', 'alloy', 'ash', 'ballad', 'echo', 'fable', 'nova', 'onyx', 'sage', 'shimmer', 'verse')
TTS1_VOICES = ('coral', 'alloy', 'ash', 'echo', 'fable', 'nova', 'onyx', 'sage', 'shimmer')
VOICES_OF = {DFLT_SPEECH: OPENAI_VOICES, 'tts-1': TTS1_VOICES, 'tts-1-hd': TTS1_VOICES}   # the voices of each OpenAI speech model
OPENAI_SPEECH = tuple(VOICES_OF)
OUT_DFLT = dict(pocket=(POCKET, 'caro_davy'), openai=(DFLT_SPEECH, 'marin'))
LOCAL_VOICES = dict(pocket=POCKET_VOICES)

`openai_voices` gives the voices of a model. A model that is not in the table gets all the voices: then OpenAI's error says what is wrong.
`fit_voice` keeps a voice that the model has, and else gives the first voice of the model.

In [ ]:
#| export
def openai_voices(model:str) -> tuple: return VOICES_OF.get(model, OPENAI_VOICES)

def fit_voice(model:str, voice:str) -> str:
    "`voice` when `model` has it, else the default voice of `model`."
    vs = openai_voices(model)
    return voice if voice in vs else vs[0]

`tts-1` has no `ballad`, so it gets `coral`. `gpt-4o-mini-tts` keeps `ballad`. An unknown model keeps its voice.

In [ ]:
test_eq(openai_voices('tts-1-hd'), TTS1_VOICES)
test_eq((fit_voice('tts-1', 'ballad'), fit_voice('tts-1', 'nova'), fit_voice(DFLT_SPEECH, 'ballad'), fit_voice('tts-9', 'ballad')), ('coral', 'nova', 'ballad', 'ballad'))

`out_resolve` fills in the default model and voice of an engine. Before 0.0.16 the local voices were `kokoro` and `hf`. Both become `pocket`.
A local voice that the engine does not have becomes its default voice, and the
model of a local engine is always its own repo. Thus an old settings file works, and nothing writes it again.
For OpenAI, a voice that the model does not have becomes the default voice of the model (`fit_voice`).
`out_current` gives the voice of the settings, with its speed. `out_where` says where the voice runs.

In [ ]:
#| export
def out_resolve(engine:str, model:str='', voice:str='') -> tuple:
    "The engine, model and voice for a name; an empty model or voice gets the default of the engine. `browser` has no model."
    if engine in ('hf', 'kokoro'): engine = 'pocket'   # the local voices before 0.0.16
    if engine not in OUT_ENGINES: raise ValueError(f'unknown voice engine {engine!r}')
    if engine == 'browser': return 'browser', '', ''
    m, v = OUT_DFLT[engine]
    if engine in LOCAL_VOICES: return engine, m, voice if voice in LOCAL_VOICES[engine] else v
    m = model or m
    return engine, m, fit_voice(m, voice or v)

def out_current() -> tuple:
    "The voice in use: `(engine, model, voice, speed)` from the settings."
    return *out_resolve(val('voice.out'), val('voice.out_model'), val('voice.out_voice')), val('voice.out_speed')

def out_where(engine:str) -> str: return 'hosted' if engine == 'openai' else 'on this Mac'

The settings give Pocket with `caro_davy`. An old Hugging Face setting or an old Kokoro setting gives the same. The old model and voice stay in the file.
OpenAI keeps its voice when its model has it: `tts-1-hd` with `ballad` speaks with `coral`. An unknown engine is an error.

In [ ]:
test_eq(out_current(), ('pocket', POCKET, 'caro_davy', 1.))
test_eq(out_resolve('pocket', '', 'mary'), ('pocket', POCKET, 'mary'))
for old in (dict(out='kokoro', out_model='mlx-community/Kokoro-82M-bf16', out_voice='bm_george'), dict(out='hf', out_model='mlx-community/csm-1b', out_voice='')):
    update_settings('voice', **old)
    test_eq(out_current()[:3], ('pocket', POCKET, 'caro_davy'))
test_eq(out_resolve('kokoro', '', 'bm_george'), ('pocket', POCKET, 'caro_davy'))
update_settings('voice', out='pocket', out_model='', out_voice='')
test_eq(out_resolve('openai', '', 'coral'), ('openai', DFLT_SPEECH, 'coral'))
test_eq(out_resolve('openai', 'tts-1-hd', 'ballad'), ('openai', 'tts-1-hd', 'coral'))
test_fail(lambda: out_resolve('espeak'), contains='unknown')

#### The time of each word

The board lights each word when the voice says it. Pocket TTS and OpenAI give no word times. Thus `by_letters` shares the length of the
audio among the words by their letters.

In [ ]:
#| export
def by_letters(text:str, secs:float) -> list:
    "A start time in seconds for each word of `text` in audio of `secs` seconds: each word takes a share by its letters."
    ws = text.split()
    n, t, out = sum(len(w) + 1 for w in ws) or 1, 0., []
    for w in ws: out.append(round(t, 3)); t += secs * (len(w) + 1) / n
    return out

Two words of 2 and 6 letters in 1 second: the first word starts at 0, the second after 3/10 of the second.

In [ ]:
test_eq(by_letters('is hidden', 1.), [0., .3])
test_eq(by_letters('', 1.), [])

#### WAV

`to_wav` makes WAV bytes (mono, 16 bit) from the float samples of a model. `clean_wav` writes the header of a WAV again with its true length.
OpenAI streams its WAV, so its header can give a wrong length.

In [ ]:
#| export
def to_wav(samples, rate:int) -> bytes:
    "WAV bytes (mono, 16 bit) for float samples from -1 to 1."
    import io, numpy as np
    a = (np.clip(np.asarray(samples, dtype=np.float32).reshape(-1), -1, 1) * 32767).astype('<i2')
    b = io.BytesIO()
    with wave.open(b, 'wb') as w: w.setnchannels(1); w.setsampwidth(2); w.setframerate(rate); w.writeframes(a.tobytes())
    return b.getvalue()

def clean_wav(audio:bytes) -> bytes:
    "`audio` with a header that gives the true length of its samples."
    import io
    with wave.open(io.BytesIO(audio)) as w: ps, frames = w.getparams(), w.readframes(len(audio))
    b = io.BytesIO()
    with wave.open(b, 'wb') as o: o.setparams(ps._replace(nframes=0)); o.writeframes(frames)
    return b.getvalue()

One second of samples at 24 kHz is one second of WAV. A header that says the data is very long becomes a header with the true length.

In [ ]:
test_eq(wav_seconds(to_wav([0.] * 24000, 24000)), 1.)
import struct
w = to_wav([0.] * 2400, 24000)
bad = w[:4] + struct.pack('<I', 0xFFFFFFFF) + w[8:40] + struct.pack('<I', 0xFFFFFFFF) + w[44:]   # as a stream: the RIFF and data sizes are unknown
test_eq(wav_seconds(clean_wav(bad)), .1)

#### A gap at the end

A local voice ends its line with no gap. On a Bluetooth or AirPlay output the last part of the sound is still in the buffer when the page
stops the audio at its `ended` event, so the line seems to stop too soon. `pad_wav` fades out the last `PAD.fade` seconds and adds
`PAD.tail` seconds of silence after the line, and `PAD.lead` seconds before it, for an output that wakes up late. It also gives where the
sound starts and ends in the new audio: Pocket TTS makes about half a second of quiet audio before and after the words, and the word times
must fall on the words, not on the quiet parts.

In [ ]:
#| export
PAD = AttrDict(lead=.05, tail=.35, fade=.03)   # seconds of silence before and after a local line, and of the fade-out at its end
LOUD = .02   # a sample above this (of full scale) is sound, not quiet

def pad_wav(audio:bytes, lead:float=PAD.lead, tail:float=PAD.tail, fade:float=PAD.fade) -> tuple:
    "`audio` with a fade-out and silence around it, and the start and end of its sound in seconds: `(wav, (start, end))`."
    import io, numpy as np
    with wave.open(io.BytesIO(audio)) as w: r, a = w.getframerate(), np.frombuffer(w.readframes(w.getnframes()), '<i2') / 32767
    a = a.astype(np.float32)
    if (n := min(len(a), int(r * fade))): a[-n:] *= np.linspace(1, 0, n, dtype=np.float32)
    loud = np.nonzero(np.abs(a) > LOUD)[0]
    on, off = (loud[0], loud[-1] + 1) if len(loud) else (0, len(a))
    wav = to_wav(np.concatenate([np.zeros(int(r * lead), np.float32), a, np.zeros(int(r * tail), np.float32)]), r)
    return wav, (round(lead + on / r, 3), round(lead + off / r, 3))

One second of sound gets 0.4 s more. The tail is silent, and the last sample of the sound is faded to zero. The sound starts after the
lead, and ends where it ended.

In [ ]:
import numpy as np

In [ ]:
w, (t0, t1) = pad_wav(to_wav(np.full(24000, .5, np.float32), 24000))
test_eq(round(wav_seconds(w), 3), 1. + PAD.lead + PAD.tail)
with wave.open(__import__('io').BytesIO(w)) as f: s = np.frombuffer(f.readframes(f.getnframes()), '<i2')
test_eq((abs(s[-int(24000 * PAD.tail):]).max(), s[int(24000 * PAD.lead) + 23999], abs(s[:int(24000 * PAD.lead)]).max()), (0, 0, 0))
test_eq((t0, t1 <= PAD.lead + 1.), (PAD.lead, True))

#### Pocket TTS

`_pocket` loads the model one time into the `tts` slot of `resident`, and keeps the state of each voice that it used. `_pocket_say` makes
the audio of one line. `normalize=True` makes the voices equally loud: some, such as `stuart_bell` and `peter_yearsley`, are quiet.
Pocket TTS has no speed, so the speed setting is only for OpenAI. When pocket-tts-mlx is not installed, the error (`POCKET_MISSING`) says how to install it, and that the browser speaks until then.

In [ ]:
#| export
POCKET_MISSING = 'Pocket TTS is not installed. Install it with: pip install "abhyasa[voice]". Until then, the browser speaks.'

def _pocket_load():
    "The Pocket TTS model (English), and an empty dict for the states of its voices."
    try: from pocket_tts_mlx import TTSModel
    except ImportError: raise RuntimeError(POCKET_MISSING) from None
    return TTSModel.load_model(), {}

def _pocket(model, voice):
    "The loaded Pocket TTS model and the state of `voice`; each loads one time."
    m, states = resident(('tts', model), _pocket_load, slot='tts')
    if voice not in states: states[voice] = m.get_state_for_audio_prompt(voice)
    return m, states[voice]

def _pocket_say(text, model, voice, speed, instructions=''):
    m, st = _pocket(model, voice)
    return to_wav(m.generate_audio(st, text, normalize=True), m.sample_rate)

#### OpenAI

The `openai` engine posts the line to the OpenAI speech API and asks for WAV. `instructions` can set the tone ("calm, like a teacher"); only
the `gpt-4o` models read it. The owner's rule is that each text that goes to a hosted model passes ramabana's guard
(`guard`, with `ramabana.guard.outgoing`). It masks a secret and refuses personal data when the privacy setting says so.

When the call fails, OpenAI sends JSON with `error.message`. `openai_why` makes it one plain line that says what to do: a voice that the
model does not have, a model that OpenAI does not know, a bad key (401), no credit or too many requests (429), or an error on OpenAI's side.
The engine raises `SpeakError` with this line, so the learner never sees the raw text of the HTTP client.

In [ ]:
#| export
OPENAI_SPEECH_URL = 'https://api.openai.com/v1/audio/speech'
OPENAI_SPEC = AttrDict(name='openai', local=False)   # the route for the guard: a model off this Mac

class SpeakError(RuntimeError):
    "A natural voice did not say a line. The text is one plain line for the learner."

def openai_why(r, model:str='', voice:str='') -> str:
    "One plain line for the failed OpenAI speech call `r`, from the `error.message` of its JSON body."
    try: err = r.json().get('error') or {}
    except Exception: err = {}
    if not isinstance(err, dict): err = dict(message=str(err))
    msg, code, s = str(err.get('message') or ''), str(err.get('code') or ''), r.status_code
    low, v = f'{code} {msg}'.lower(), voice.lower()
    if s == 401: return 'OpenAI: the key does not work. Set a new key on the Settings page, under More in Speech to text.'
    if s == 429 and 'quota' in low: return 'OpenAI: your account has no credit. Add credit at platform.openai.com, or pick another voice.'
    if s == 429: return 'OpenAI: too many requests. Wait one minute, then try again.'
    if s >= 500: return f'OpenAI did not answer (error {s}). Try again later.'
    if voice not in openai_voices(model) or 'voice' in low or (v and v in low):
        return f'OpenAI: the voice {voice.title()} does not work with {model}. Pick another voice or the model {DFLT_SPEECH}.'
    if s == 404 or 'model' in low: return f'OpenAI: the model {model} does not work for speech with your key. Pick the model {DFLT_SPEECH}.'
    if msg: return f"OpenAI did not say the line: {msg.split('. ')[0].rstrip('.')}."
    return f'OpenAI did not say the line (error {s}).'

def _openai_say(text, model, voice, speed, instructions=''):
    key, _ = openai_key()
    if not key: raise SpeakError('There is no OpenAI key. Set one on the Settings page.')
    body = dict(model=model, voice=voice, input=guard(text, OPENAI_SPEC), response_format='wav', speed=speed)
    if instructions and not model.startswith('tts-1'): body['instructions'] = guard(instructions, OPENAI_SPEC)
    httpx = __import__('httpx')
    try: r = (http_post or httpx.post)(OPENAI_SPEECH_URL, headers={'Authorization': f'Bearer {key}'}, json=body, timeout=60)
    except (httpx.HTTPError, OSError) as e: raise SpeakError('OpenAI did not answer. Check the network, then try again.') from e
    if r.status_code >= 400: raise SpeakError(openai_why(r, model, voice))
    return clean_wav(r.content)

SAY = dict(pocket=_pocket_say, openai=_openai_say)

#### Maths as words

A voice reads `$ym$` as "yum", and `\alpha` as "backslash alpha". `spoken` puts the maths of a line into words before any voice says it:
`$…$`, `$$…$$`, `\(…\)`, `\[…\]`, a short name in back quotes, and a bare `W_Q` (one letter with a short subscript). `tex_words` reads a
run of single letters letter by letter, a Greek letter by its name, the common operators, `^2` as "squared", `_m` as "sub m", `\frac`,
`\sqrt`, `\bar` and `\hat`. Each table holds one kind of name. Prose does not change: "ym" with no `$` stays "ym". No small library on PyPI
does this (kosha found none in the dependencies), so it is one function with tables.

The text on the board stays as it is. The board lights one part at a time: a `$…$` is one part, and each other word is one part
(`SAY_PART`, the same rule as rupaka's board). Thus `spoken` gives, for each spoken word, the part that it reads, and `part_times` gives the
start of each part from the start of its first spoken word.

In [ ]:
#| export
import bisect
GREEK = 'alpha beta gamma delta epsilon varepsilon zeta eta theta vartheta iota kappa lambda mu nu xi omicron pi varpi rho varrho sigma varsigma tau upsilon phi varphi chi psi omega'.split()
TEX_SAY = {**{g: g.removeprefix('var') for g in GREEK}, **{g.capitalize(): 'capital ' + g for g in GREEK if not g.startswith('var')},
           'cdot': 'times', 'times': 'times', 'div': 'divided by', 'pm': 'plus or minus', 'mp': 'minus or plus',
           'le': 'is at most', 'leq': 'is at most', 'ge': 'is at least', 'geq': 'is at least', 'ne': 'is not equal to', 'neq': 'is not equal to',
           'approx': 'is about', 'sim': 'is like', 'propto': 'is proportional to', 'equiv': 'is the same as', 'infty': 'infinity',
           'partial': 'partial', 'nabla': 'nabla', 'to': 'to',
           'rightarrow': 'to', 'mapsto': 'maps to', 'in': 'in', 'notin': 'not in', 'subset': 'in', 'forall': 'for all', 'exists': 'there is',
           'ldots': 'and so on', 'cdots': 'and so on', 'dots': 'and so on', 'circ': 'after', 'otimes': 'times', 'odot': 'times', 'ell': 'l'}
TEX_BIG = dict(sum='sum', prod='product', int='integral')   # with limits: 'the sum from i equals 1 to n of'
TEX_FN = {'log', 'ln', 'exp', 'sin', 'cos', 'tan', 'max', 'min', 'arg', 'argmax', 'argmin', 'det', 'lim', 'sup', 'inf', 'softmax', 'tr', 'diag', 'var', 'mod'}   # read as words
TEX_ACCENT = dict(bar='{} bar', hat='{} hat', tilde='{} tilde', vec='vector {}', dot='{} dot', overline='{} bar', widehat='{} hat', mathbf='{}',
                  boldsymbol='{}', mathrm='{}', mathit='{}', mathcal='{}', mathbb='{}', text='{}', textrm='{}', operatorname='{}', mathsf='{}')
TEX_OPS = {'+': 'plus', '-': 'minus', '=': 'equals', '<': 'is less than', '>': 'is greater than', '/': 'over', '*': 'times', '!': 'factorial',
           "'": 'prime', '|': '', '(': '', ')': '', '[': '', ']': '', '&': '', '~': '', ':': '', ';': ''}
TEX_POW = {'2': 'squared', '3': 'cubed', 'T': 'transpose', '-1': 'inverse', 'minus 1': 'inverse', 'top': 'transpose', 'prime': 'prime'}
_TEX_TOK = re.compile(r'\\[A-Za-z]+|\\.|[A-Za-z]+|\d+(?:\.\d+)?|\S')

def tex_words(tex:str) -> str:
    "TeX maths as words to say: `y_m` is 'y sub m', `\\alpha^2` is 'alpha squared', `\\frac{a}{b}` is 'a over b'."
    ts, i = _TEX_TOK.findall(tex), 0
    def arg():
        "The words of the next group `{...}`, or of one atom."
        nonlocal i
        if i >= len(ts): return []
        if ts[i] == '{':
            i += 1; out = []
            while i < len(ts) and ts[i] != '}': out += item()
            i += 1
            return out
        return item(scripts=False)
    def item(scripts=True):
        nonlocal i
        t = ts[i]; i += 1
        if t == '{': i -= 1; out = arg()
        elif t in ('}', '^', '_'): out = []
        elif t.startswith('\\'):
            c = t[1:]
            if c == 'frac' or c == 'dfrac' or c == 'tfrac': a = arg(); out = [*a, 'over', *arg()]
            elif c == 'sqrt':
                n = []
                if i < len(ts) and ts[i] == '[':
                    i += 1
                    while i < len(ts) and ts[i] != ']': n += item()
                    i += 1
                out = ['the', *({'3': ['cube']}.get(' '.join(n), [*n, 'th'] if n else ['square'])), 'root', 'of', *arg()]
            elif c in TEX_BIG:
                lim = {}
                while i < len(ts) and ts[i] in ('^', '_'): s = ts[i]; i += 1; lim[s] = arg()
                out = ['the', TEX_BIG[c], *(['from', *lim['_']] if '_' in lim else []), *(['to', *lim['^']] if '^' in lim else []), 'of']
            elif c in TEX_ACCENT: out = TEX_ACCENT[c].format(' '.join(arg())).split()
            elif c in TEX_SAY: out = TEX_SAY[c].split()
            elif c in TEX_FN: out = [c]
            elif c in ('left', 'right', 'big', 'Big', 'bigg', 'quad', 'qquad', ',', ';', '!', ' ', 'displaystyle', 'limits'): out = []
            elif c in ('{', '}', '|', '\\'): out = []
            else: out = [c]
        elif t[0].isalpha(): out = [t] if len(t) > 3 or t in TEX_FN else list(t)   # ym is 'y m'; a long run is a word
        elif t[0].isdigit(): out = [t]
        else: out = [TEX_OPS.get(t, t)] if TEX_OPS.get(t, t) else []
        while scripts and i < len(ts) and ts[i] in ('^', '_'):
            s = ts[i]; i += 1; a = arg(); j = ' '.join(a)
            if s == '_': out += ['sub', *a]
            elif j in TEX_POW: out.append(TEX_POW[j])
            else: out += ['to the', *a] if len(a) == 1 else ['to the power', *a]
        return out
    out = []
    while i < len(ts): out += item()
    return ' '.join(w for w in out if w)

SAY_PART = re.compile(r'\$([^$]+)\$([.,;:!?)]*)|(\S+)')   # the parts of a line as rupaka's board shows them: a `$...$` or a word
_MATH = re.compile(r'\$\$(?P<a>.+?)\$\$|\$(?P<b>[^$]+)\$|\\\((?P<c>.+?)\\\)|\\\[(?P<d>.+?)\\\]|`(?P<e>[A-Za-z][A-Za-z0-9_^{}]{0,11})`'
                   r'|(?<!\S)(?P<f>[A-Za-z]_(?:\{[^{}\s]{1,12}\}|[A-Za-z0-9]{1,3}))(?=[.,;:!?)]*(?:\s|$))')

def part_times(times:list, at:list, n:int) -> list:
    "The start of each of the `n` parts of a line, from the start of each spoken word and the part that it reads."
    out, last = [], times[0] if times else 0.
    for k in range(n):
        last = next((t for t, a in zip(times, at) if a == k), last)   # a part with no spoken word of its own starts with the part before
        out.append(last)
    return out

def spoken(text:str) -> tuple:
    "The words to say for a line with maths, and for each spoken word the index of the part of the line that it reads."
    out, src = [], []   # the spoken characters, and the place in `text` of each
    at = 0
    for m in _MATH.finditer(text):
        out.append(text[at:m.start()]); src += range(at, m.start())
        w = tex_words(next(g for g in m.groups() if g is not None))
        out.append(w); src += [m.start()] * len(w); at = m.end()
    out.append(text[at:]); src += range(at, len(text))
    s = ''.join(out)
    starts = [p.start() for p in SAY_PART.finditer(text)]
    words = [(w.group(), max(0, bisect.bisect_right(starts, src[w.start()]) - 1)) for w in re.finditer(r'\S+', s)]
    return ' '.join(w for w, _ in words), [k for _, k in words]

The examples of the owner. Prose with "rhythm" and a bare "ym" stays as it is, and each prose word is its own part.

In [ ]:
test_eq([spoken(s)[0] for s in ('$y_m$', '$ym$', r'$\alpha^2$', r'$\frac{a}{b}$')], ['y sub m', 'y m', 'alpha squared', 'a over b'])
test_eq(spoken('The rhythm of ym stays.'), ('The rhythm of ym stays.', [0, 1, 2, 3, 4]))
test_eq(spoken(r'Here $\hat{y} = \sqrt{x^2 + 1}$, and W_Q is a matrix.')[0], 'Here y hat equals the square root of x squared plus 1, and W sub Q is a matrix.')
test_eq(spoken(r'Use \(\bar{x}_i\) and `ym` now.')[0], 'Use x bar sub i and y m now.')
test_eq(spoken(r'$$\sum_{i=1}^{n} x_i^{-1}$$')[0], 'the sum from i equals 1 to n of x sub i inverse')
test_eq(spoken(r'$\mathbf{W}^T x$ and $\Sigma$')[0], 'W transpose x and capital sigma')

`$q = xW_Q$` is one part on the board, so its five spoken words all read part 1. A part with no word of its own starts with the part before it.

In [ ]:
s, at = spoken('So $q = xW_Q$ here.')
test_eq((s, at), ('So q equals x W sub Q here.', [0, 1, 1, 1, 1, 1, 1, 2]))
test_eq(part_times([0., .1, .2, .3, .4, .5, .6, .7], at, 3), [0., .1, .7])
test_eq(part_times([0., .5], [0, 2], 3), [0., 0., .5])

#### speak

`speak` gives the WAV bytes of one line and the start time of each of its words. Without an engine, it uses the voice of the settings.
For the browser voice, it refuses: the page speaks then.
The voice says the words of `spoken`, and the times are for the parts of the line as the board shows them.
A local line gets its gap (`pad_wav`), and its words share the time of its sound. An OpenAI line keeps its audio as it is.

In [ ]:
#| export
def speak(text:str, engine:str=None, model:str=None, voice:str=None, speed:float=None, instructions:str='') -> tuple:
    "WAV bytes for `text` with its maths in words, and the start time in seconds of each part of `text` (`SAY_PART`)."
    e, m, v, s = out_current()
    e, m, v = out_resolve(engine, model or '', voice or '') if engine else out_resolve(e, model or m, voice or v)
    if e == 'browser': raise ValueError('The browser speaks with its own voice. There is no audio here.')
    say, at = spoken(text)
    wav = SAY[e](say, m, v, float(speed or s), instructions)
    if e == 'openai': t0, t1 = 0., wav_seconds(wav)
    else: wav, (t0, t1) = pad_wav(wav)
    return wav, part_times([round(t0 + t, 3) for t in by_letters(say, t1 - t0)], at, len(SAY_PART.findall(text)))

#### Tests with a fake model

A fake Pocket TTS: it gives 0.25 s of silence for each word, and it notes each voice state that it makes. The loader is a fake too, so the test downloads nothing.

In [ ]:
import numpy as np

In [ ]:
class _Pocket:
    sample_rate = 24000
    def __init__(self): self.said, self.voices = [], []
    def get_state_for_audio_prompt(self, v): self.voices.append(v); return dict(voice=v)
    def generate_audio(self, st, text, normalize=False):
        self.said.append((st['voice'], text, normalize))
        return np.zeros(6000 * len(text.split()), np.float32)
loads = []
_real_load, _pocket_load = _pocket_load, (lambda: loads.append(1) or (_Pocket(), {}))

`speak` gives one second of WAV for four words, with the gap around it, and a time for each word that goes up. The second line does not load the model again,
and a voice state loads one time.

In [ ]:
wav, words = speak('Noise goes in, slowly.', 'pocket')
test_eq(round(wav_seconds(wav), 3), 1. + PAD.lead + PAD.tail)   # one second of words, and the gap
test_eq(words, [round(PAD.lead + t, 3) for t in by_letters('Noise goes in, slowly.', 1.)])
test_eq(words, sorted(words))
speak('Again.', 'pocket', voice='mary'); speak('Once more.', 'pocket', voice='mary')
pk = _slots['tts'][0].obj[0]
test_eq((loads, pk.voices, pk.said[-1]), ([1], ['caro_davy', 'mary'], ('mary', 'Once more.', True)))

The settings choose the voice when `speak` has no engine. The browser voice makes no audio here.

In [ ]:
update_settings('voice', out='pocket', out_voice='paul')
speak('hello')
test_eq(pk.said[-1][0], 'paul')
update_settings('voice', out='browser', out_voice='')
test_fail(lambda: speak('hello'), contains='browser')

The OpenAI engine refuses with no key. With a key, it sends the text after the guard, with the model, the voice and the tone. Here a fake guard puts a mark on each text, so the test sees that each text passed it.

In [ ]:
real_guard, guard = guard, lambda s, model=None, mode=None: f'[{model.name}] {s}'
test_fail(lambda: speak('hello', 'openai'), contains='no OpenAI key')
set_openai_key('sk-test-1234')
class _W:
    status_code, content = 200, to_wav([0.] * 12000, 24000)
sent = {}
http_post = lambda url, **kw: sent.update(url=url, **kw) or _W()
wav, words = speak('Look at the noise.', 'openai', instructions='calm, like a teacher')
test_eq(sent['url'], OPENAI_SPEECH_URL)
test_eq(sent['json'], dict(model=DFLT_SPEECH, voice='marin', input='[openai] Look at the noise.', response_format='wav', speed=1., instructions='[openai] calm, like a teacher'))
test_eq((wav_seconds(wav), len(words)), (.5, 4))

A fake OpenAI that fails with a status and an `error` body. `_fail` gives the line that the learner sees.

In [ ]:
class _Bad:
    def __init__(self, status, message='', code=None): self.status_code, self.body = status, dict(error=dict(message=message, type='invalid_request_error', code=code))
    def json(self): return self.body
def _fail(r, model=DFLT_SPEECH, voice='marin'):
    global http_post
    good, http_post = http_post, (lambda url, **kw: r)
    try: speak('Hi.', 'openai', model, voice)
    except SpeakError as e: return str(e)
    finally: http_post = good

A voice that the model does not have gives one line with the fix. An unknown model, a bad key, no credit, too many requests and an error of
OpenAI each give their own line. (`speak` itself sends `coral` for `ballad` with `tts-1-hd`: see `out_resolve`.) No line has the raw text of the HTTP client or a link.

In [ ]:
test_eq(openai_why(_Bad(400, "Invalid voice 'ballad' for model 'tts-1-hd'."), 'tts-1-hd', 'ballad'),
        'OpenAI: the voice Ballad does not work with tts-1-hd. Pick another voice or the model gpt-4o-mini-tts.')
test_eq(openai_why(_Bad(400, 'Invalid value for voice.'), DFLT_SPEECH, 'verse'), 'OpenAI: the voice Verse does not work with gpt-4o-mini-tts. Pick another voice or the model gpt-4o-mini-tts.')   # the table can be old: the message says so
test_eq(_fail(_Bad(404, 'The model `tts-9` does not exist.', 'model_not_found'), 'tts-9', 'nova'),
        'OpenAI: the model tts-9 does not work for speech with your key. Pick the model gpt-4o-mini-tts.')
test_eq(_fail(_Bad(401, 'Incorrect API key provided.', 'invalid_api_key')), 'OpenAI: the key does not work. Set a new key on the Settings page, under More in Speech to text.')
test_eq(_fail(_Bad(429, 'You exceeded your current quota.', 'insufficient_quota')), 'OpenAI: your account has no credit. Add credit at platform.openai.com, or pick another voice.')
test_eq(_fail(_Bad(429, 'Rate limit reached.', 'rate_limit_exceeded')), 'OpenAI: too many requests. Wait one minute, then try again.')
test_eq(_fail(_Bad(503, 'busy')), 'OpenAI did not answer (error 503). Try again later.')
test_eq(_fail(_Bad(400, 'Input is too long. Shorten it.')), 'OpenAI did not say the line: Input is too long.')
for s in [_fail(_Bad(c, 'x')) for c in (400, 401, 429, 503)]: assert 'Client error' not in s and 'http' not in s

`tts-1` does not read `instructions`, so the engine does not send them. Then the test puts back the real guard, the real HTTP client and the real loader.

In [ ]:
speak('Hi.', 'openai', 'tts-1', instructions='calm')
assert 'instructions' not in sent['json']
guard, http_post = real_guard, None
set_openai_key('')
_pocket_load = _real_load
unload('tts')

#### One worker for the voice

`Speaker` has one worker thread for all the calls of the speaking voice, separate from the speech to text worker. Thus a line of the board
does not wait for a segment of dictation. It keeps the lines that it said in this run (at most `most`), by engine, model, voice, speed and
text. The board asks for the same line again when the learner goes back a step, and that line then comes at once.
Two requests for the same line share one call. `warm` loads the model in use with one short line, so the first line of a board does not
wait for the load. The browser voice and OpenAI have nothing to load.

A model that is not on this Mac yet is not downloaded for a line: the Settings page shows its download. Until then, `say` refuses the line
with the word "browser", as for the browser voice, so the page and the Mac app speak with their own voice.

When the voice fails on a line, `error` keeps why, as one plain line, so the Settings page can show it under the choice. The next line that
the voice says clears it.

In [ ]:
#| export
class Speaker:
    "One worker thread for the speaking voice, and the lines that it said in this run."
    def __init__(self, fn=None, most:int=64, here=None):
        self.fn, self.most, self.lines, self.status, self.warmed, self.error = fn or speak, most, {}, 'not loaded', None, ''
        self.here = here or (lambda m: cached(m))   # a model that is not on this Mac is not downloaded here: the Settings page gets it
        self.ex, self.lock = CtxPool(1, thread_name_prefix='voice'), threading.Lock()

    def ready(self, engine:str, model:str):
        "Raise when the model of the local voice is not on this Mac yet: the browser speaks until then."
        if engine in LOCAL_VOICES and not self.here(model): raise ValueError('The voice model is not on this Mac yet, so the browser speaks. Download it on the Settings page.')

    def key(self, text:str, engine:str=None, model:str=None, voice:str=None, speed:float=None) -> tuple:
        "The key of a line: `(engine, model, voice, speed, text)`, with the settings for what is not given."
        e, m, v, s = out_current()
        e, m, v = out_resolve(engine, model or '', voice or '') if engine else out_resolve(e, model or m, voice or v)
        return e, m, v, float(speed or s), ' '.join(str(text or '').split())

    def say(self, text:str, **kw) -> dict:
        "The audio of one line: `dict(wav, words, engine, model, voice, speed, where, cached)`."
        k = self.key(text, **kw)
        if k[0] == 'browser': raise ValueError('The browser speaks with its own voice. There is no audio here.')
        if not k[4]: raise ValueError('There is no text to say.')
        self.ready(*k[:2])
        with self.lock:
            hit = k in self.lines
            f = self.lines.pop(k) if hit else self.ex.submit(self.fn, k[4], *k[:4])
            self.lines[k] = f   # the newest at the end
            while len(self.lines) > self.most: self.lines.pop(next(iter(self.lines)))
        try: wav, words = f.result()
        except Exception as e:
            with self.lock: self.lines.pop(k, None)   # a failed line is not kept, so the next request tries again
            if not isinstance(e, (ValueError, Withheld)): self.error = str(e)   # the voice failed: the Settings page says why
            raise
        self.error = ''
        return dict(wav=wav, words=words, engine=k[0], model=k[1], voice=k[2], speed=k[3], where=out_where(k[0]), cached=hit)

    def warm(self):
        "Load the voice in use on the worker with one short line; None when there is nothing to load."
        e, m, v, s = out_current()
        if e in ('browser', 'openai'): self.status = 'ready'; return None
        if self.warmed == (e, m) and self.status != 'loading' or not self.here(m): return None
        self.status, self.warmed = 'loading', (e, m)
        def go():
            try: self.fn('Ready.', e, m, v, s); self.status = 'ready'
            except Exception as ex: self.status, self.warmed = f'failed: {ex}', None
        return self.ex.submit(go)

A fake voice counts its calls. The same line comes from the cache, and a new speed is a new line.

In [ ]:
calls = []
def fake_speak(text, engine, model, voice, speed):
    calls.append((text, engine, voice, speed)); time.sleep(.02)
    return to_wav([0.] * 2400, 24000), by_letters(text, .1)
sk = Speaker(fake_speak, most=2, here=lambda m: True)
test_fail(lambda: sk.say('hi', engine='browser'), contains='browser')
r = sk.say('Look  here.', engine='pocket')
test_eq((r['cached'], r['where'], r['voice'], r['words']), (False, 'on this Mac', 'caro_davy', by_letters('Look here.', .1)))
test_eq(sk.say('Look here.', engine='pocket')['cached'], True)
sk.say('Look here.', engine='pocket', speed=1.5)
test_eq(len(calls), 2)

Two requests at the same time for a new line make one call. The cache keeps at most `most` lines: the oldest goes first.

In [ ]:
with ThreadPoolExecutor(2) as ex: list(ex.map(lambda _: sk.say('Both at once.', engine='pocket'), range(2)))
test_eq(len(calls), 3)
test_eq(len(sk.lines), 2)
test_eq(sk.say('Look here.', engine='pocket')['cached'], False)

A failed line is not kept, and `error` says why until a line works. `warm` loads the voice of the settings one time, and does nothing for the browser voice.
A model that is not here is not warmed, and a line for it says that the browser speaks.

In [ ]:
def _flaky(text, *a):
    if text == 'x': raise SpeakError('OpenAI: too many requests. Wait one minute, then try again.')
    return fake_speak(text, *a)
sk = Speaker(_flaky, here=lambda m: True)
test_fail(lambda: sk.say('x', engine='pocket'))
test_eq((sk.lines, sk.error), ({}, 'OpenAI: too many requests. Wait one minute, then try again.'))
sk.say('y', engine='pocket'); test_eq(sk.error, '')
update_settings('voice', out='browser')
test_is(Speaker(fake_speak).warm(), None)
update_settings('voice', out='pocket')
sk = Speaker(fake_speak, here=lambda m: True); calls.clear()
sk.warm().result(); test_is(sk.warm(), None)
test_eq((sk.status, calls), ('ready', [('Ready.', 'pocket', 'caro_davy', 1.)]))
sk = Speaker(fake_speak, here=lambda m: False)
test_is(sk.warm(), None)
test_fail(lambda: sk.say('x'), contains='browser')

### Segments in order

The browser sends one segment each time you stop to think. `Speech` has one worker thread for all the model calls.
Thus only one model is in memory, and the segments run one at a time, in the order they came.
A `Dictation` collects the segments of one answer. `next` gives the text in the order of the segment numbers,
also when a slow request arrives late. When the browser sends a segment again (after an error), the code does not use it two times.

In [ ]:
#| export
class Dictation:
    "The segments of one spoken answer: each is transcribed on the speech worker, and the text comes out in order."
    def __init__(self, speech, hint=''):
        store_attr()
        self.seen, self.ready, self.out, self.n_done, self.total, self.t = set(), {}, 0, 0, None, time.time()
        self.cv = threading.Condition()

    def add(self, seq:int, audio:bytes) -> bool:
        "Queue segment `seq`; False when it came before."
        with self.cv:
            if seq in self.seen: return False
            self.seen.add(seq)
        self.speech.submit(self._run, seq, audio)
        return True

    def _run(self, seq, audio):
        try: r = dict(seq=seq, text=self.speech.fn(audio, hint=self.hint).strip())
        except Exception as e: r = dict(seq=seq, text='', error=str(e))
        with self.cv: self.ready[seq] = r; self.n_done += 1; self.cv.notify_all()

    def stop(self, n:int):
        "There are `n` segments in all."
        with self.cv: self.total = n; self.cv.notify_all()

    @property
    def pending(self) -> int: return len(self.seen) - self.n_done

    def next(self, timeout=None):
        "The next result in order; `dict(done=True)` after the last one; None at the timeout."
        with self.cv:
            if not self.cv.wait_for(lambda: self.out in self.ready or self.total == self.out, timeout): return None
            if self.out not in self.ready: return dict(done=True)
            self.out += 1
            return self.ready.pop(self.out - 1)

`Speech` owns the worker. `warm` loads the model in use with half a second of silence, so the first segment does not wait for it.
`use` changes the engine and model, saves the change, and warms the new model.

In [ ]:
#| export
class Speech:
    "One worker thread for every speech model call, and the open dictations."
    def __init__(self, fn=None):
        self.fn, self.runs, self.status = fn or transcribe, {}, 'not loaded'
        self.ex = CtxPool(1, thread_name_prefix='speech')

    def submit(self, f, *a, **kw): return self.ex.submit(f, *a, **kw)

    def warm(self):
        "Load the model in use on the worker."
        def go():
            try: self.fn(silence(), hint=''); self.status = 'ready'
            except Exception as ex: self.status = f'failed: {ex}'
        self.status = 'loading'   # at once, so a page drawn now shows it
        return self.submit(go)

    def use(self, engine:str, model:str=''):
        "Switch to `engine` and `model`, save the choice, and warm the new model."
        engine, model = resolve(engine, model)
        update_settings('stt', engine=engine, model=model)
        return self.warm()

    def start(self, hint:str='') -> str:
        "Open a dictation; returns its id. Dictations older than an hour are dropped."
        for k in [k for k, d in self.runs.items() if time.time() - d.t > 3600]: self.runs.pop(k, None)
        did = uuid.uuid4().hex[:10]
        self.runs[did] = Dictation(self, hint)
        return did

#### Tests with a fake engine

This fake engine is slow on even segments, so the requests finish out of order. The text must still come out in order.

In [ ]:
calls = []
def slow_fake(audio, hint=''):
    n = int(audio.decode())
    calls.append(n)
    time.sleep(.05 if n % 2 == 0 else 0)
    return f'segment {n} ({hint})'
sp = Speech(slow_fake)
d = sp.runs[sp.start('hetu')]

Segments 0 to 5 arrive out of order, from several threads. Segment 3 comes two times.

In [ ]:
order = [2, 0, 1, 3, 3, 5, 4]
with ThreadPoolExecutor(4) as ex: added = list(ex.map(lambda n: d.add(n, str(n).encode()), order))
test_eq(sum(added), 6)
d.stop(6)
out = []
while not (r := d.next(5)).get('done'): out.append(r)
test_eq([r['seq'] for r in out], list(range(6)))
test_eq(out[0]['text'], 'segment 0 (hetu)')
test_eq(sorted(calls), list(range(6)))

All the model calls ran on the one worker thread, one at a time.

In [ ]:
seen = set()
sp.submit(lambda: seen.add(threading.current_thread().name)).result()
test_eq(len(seen), 1)
test_eq(d.pending, 0)

An engine error gives an empty text with the error. It does not stop the segments after it.

In [ ]:
def flaky(audio, hint=''):
    if audio == b'bad': raise RuntimeError('no model')
    return 'ok'
sp = Speech(flaky)
d = sp.runs[sp.start()]
d.add(0, b'bad'); d.add(1, b'good'); d.stop(2)
test_eq(d.next(5), dict(seq=0, text='', error='no model'))
test_eq(d.next(5), dict(seq=1, text='ok'))
test_eq(d.next(5), dict(done=True))

Without `stop`, `next` waits and then gives None. `use` saves the choice and warms the new model with silence.

In [ ]:
d = sp.runs[sp.start()]
test_is(d.next(.05), None)
heard = []
sp = Speech(lambda a, hint='': heard.append(wav_seconds(a)) or '')
sp.use('parakeet').result()
test_eq(current(), ('mlx', DFLT_PARAKEET))
test_eq(heard, [.5])
assert sp.status.startswith('ready'), sp.status
update_settings('stt', engine='mlx', model='')

### Real models

These tests use a real clip and real models. They are slow. Run them with `nbdev-test --flags slow`.

In [ ]:
#| slow
clip = first(p for p in (Path('../../rishi/nbs/speech.wav'), Path('../../../rishi/nbs/speech.wav')) if p.exists())   # from the repo or a worktree
if clip:
    for e in ('mlx', 'whisper'): assert transcribe(clip, e, hint='Tennyson').strip(), e
    unload()

In [ ]:
#| slow
wav, words = speak('The forward process adds a little noise at each step.', 'pocket')
assert wav_seconds(wav) > 1 and words, words
unload('tts')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()